# 🤖 02 · 모델 실행 및 실험 평가 — 학습 없이 Generate + 종합·질문별 RAGAS
**01_rag_build.ipynb 에서 만든 인덱스(FAISS · BM25 · 임베딩 모델)를 그대로 불러옵니다.** 파싱·임베딩·학습은 하지 않습니다(`generate_only=True`).

```
사용자가 챗봇에 친 질문(그대로) ─▶ BM25 × Dense(FAISS) 하이브리드 검색 ─▶ gpt-5-mini 답변 ─▶ RAGAS(전체 종합 → 질문별)
```
- 질문(`0-3`)은 챗봇 입력창에 적은 문장 그대로입니다. 오타·한영 자판·한자·영어 혼용도 **전처리·가공 없이** 보냅니다. 정답이나 메타데이터도 붙이지 않습니다.
- 출력: **① 전체 질문 종합 RAGAS 표 → ② 질문별 RAGAS 표** (Faithfulness · Answer Relevancy · Context Precision · **Answer Correctness**)
- Answer Correctness 는 정답이 필요해서, 정답은 질문과 따로 `0-4` 셀에 적습니다(질문 문장은 그대로).
- 조절할 수 있는 것: temperature · top_p · max tokens · top-k · Dense:BM25 비율 · fusion · RAGAS 지표·심판 모델 (`0-2`)
- ⏸ ■ = 안전 중지(질문 1건마다 저장). 같은 셀을 다시 실행하면 이어서 진행하고, 설정·질문이 같으면 저장된 답변·점수를 재사용합니다.
- **원격 GCP(g2-standard-4 + L4)에서 실행:** 01 을 VM 에서 돌렸거나, 로컬에서 만든 `artifacts` 폴더를 VM 프로젝트 폴더에 통째로 올리면 됩니다(윈도우 경로는 `0-2` 가 자동 변환). OpenAI 키는 `.env`, HuggingFace 는 `hf auth login` 정보를 씁니다(입력창 없음).
- **토큰 절약:** `0-2`에서 답변·심판 모델과 **추론 강도(none~max)**를 직접 고르고, `EVAL_MODE="free"`(골든 데이터셋, 토큰 0)로 평가할 수 있습니다. `1-3`·`2`에서 실제 사용 토큰과 예상 비용을 보여줍니다.
- ⚠️ **노트북은 한 번에 하나만 실행하세요** (VM RAM 16GB). 다른 노트북 커널이 남아 있으면 메모리 부족으로 커널이 강제 종료됩니다. `0-1` 셀이 다른 커널과 RAM·GPU 사용량을 보여주고, `CLOSE_OTHER_KERNELS = True` 로 정리할 수 있습니다.
- **🔎 검색 파이프라인(1-1b):** ① GPT 가 질문을 검색용 질문으로 해석(오타·한영 자판·한자·영어 — 추론 강도 low~high 조절) → ② 원래 질문 + 다시 쓴 질문으로 각각 BM25+Dense 검색 후 RRF 병합 → ③ 리랭커(내 GPU) → ④ GPT 답변 — 원래 질문 + 검색용 질문(GPT 해석)을 함께 받아 답함(`ANSWER_WITH_REWRITE`).
- **🧠 생성 과정 자체 평가 (`2-2`, $0):** RAGAS 를 검색 · 생성 · E2E 단계로 나누고, 답변이 검색된 근거를 얼마나 충실히 썼는지(숫자 근거율 · 문장 근거율 · 핵심 사실 활용 · 거절 · 문서 외 답변 · 잘림)를 GPT 없이 점검합니다. '검색이 성공한 질문'만 따로 봐서 생성 단계 자체 실력을 분리합니다. 결과 → `eval/results/generation_02.json` (04_e2e_eval 이 사용)
- **📋 질문 세트(`0-3`):** 함정 질문 20개(8개 제안요청서 · 오타·띄어쓰기 / 한문 / 한영 자판 / 영어 섞임 각 5개, 그중 5개는 틀린 전제) (기본값. 01 의 RFP 정답 세트 R01~X03 은 `QUESTION_SETS` 에 "RFP" 를 넣을 때만). 함정 질문은 근거 문서가 정해져 있어 검색(정답 문서 적중) · 생성(전제 교정 · 핵심 사실)까지 채점합니다.
- **💵 비용:** GPT 를 쓰는 셀마다 그 셀에서 쓴 토큰과 달러를 출력하고, 마지막 `4)` 셀에서 단계별 합계를 보여줍니다.

## 0. 환경 · 생성 파라미터 · 질문
> 👥 **담당:** 0-1 PM · 0-2 검색 설정 = 검색 · 리랭킹, LLM·질문 해석 설정 = 질문 해석 · 답변 · 0-3~0-5 = 평가 · 데이터셋

In [1]:
# 0-1) 경로 · 토큰(OpenAI .env / HuggingFace 로그인) · CPU/GPU 버전 확인   — 보통 10~30초
#  ▸ 원격 GCP: g2-standard-4 (vCPU 4 · RAM 16GB) + NVIDIA L4 24GB — VS Code Remote-SSH / JupyterHub 커널에서 실행
#  ▸ HuggingFace 로그인(선택, VM 터미널에서 한 번): hf auth login   ← 이후 HF_TOKEN 입력창 없음
#  ▸ ⑤ 메모리 점검: 다른 노트북 커널이 RAM·GPU 를 잡고 있으면 경고 (CLOSE_OTHER_KERNELS=True 로 종료)
import os, sys, time
from pathlib import Path

T0 = time.time()
PROJECT_DIR = Path.cwd() if (Path.cwd() / "rag").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_DIR))
os.chdir(PROJECT_DIR)
if sys.version_info < (3, 10):
    raise RuntimeError(f"Python 3.10 이상이 필요합니다 (현재 {sys.version.split()[0]}) → 커널을 Python 3.10 으로 바꾸세요")

HF_CACHE_DIR = None        # 모델 저장 위치(기본 ~/.cache/huggingface, 약 10GB). 디스크가 부족하면 예) "/mnt/disk/hf_cache"
NEED_OPENAI  = True       # 02: 답변 생성·RAGAS 에 필수

print("① torch 로딩…", flush=True)
from rag.hw import apply_runtime_env, detect, gpu_mem, summary, require_gpu
apply_runtime_env(cache_dir=HF_CACHE_DIR)
print(f"   완료 {time.time() - T0:.1f}s", flush=True)

print("② 토큰 — OpenAI: .env / HuggingFace: 로그인 정보(hf auth login) 사용, HF_TOKEN 입력창 없음", flush=True)
from rag.credentials import load_tokens, verify_tokens, mask
import pandas as pd
tok_status = load_tokens(PROJECT_DIR / ".env", ask=False, save_to_env=False, required=())   # 입력창을 띄우지 않음
_k = os.environ.get("OPENAI_API_KEY", "").strip()
while _k.startswith("sk-sk-"):                    # 'sk-' 가 두 번 붙은 키 바로잡기 (메모리에서만)
    _k = _k[3:]
if _k:
    os.environ["OPENAI_API_KEY"] = _k
elif NEED_OPENAI:
    import getpass
    print("🔑 OPENAI_API_KEY 가 .env 에 없습니다 → 입력창(VS Code: 창 맨 위)에 붙여넣고 Enter", flush=True)
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ").strip()
    tok_status["OPENAI_API_KEY"]["출처"] = "입력(메모리에만)"

# HuggingFace: 터미널에서 한 번 'hf auth login'(구버전: huggingface-cli login) 하면 저장된 로그인 정보를 자동 사용
from huggingface_hub import get_token
_hf = get_token() or next((p.read_text().strip() for p in [Path.home() / ".cache" / "huggingface" / "token"] if p.exists()), None)
if _hf:
    os.environ["HF_TOKEN"] = _hf
oa = os.environ.get("OPENAI_API_KEY", "")
display(pd.DataFrame({
    "OPENAI_API_KEY": {"값": mask(oa), "출처": tok_status["OPENAI_API_KEY"]["출처"] if oa else "-",
                       "용도": "gpt-5-mini 생성·RAGAS 심판" + ("" if NEED_OPENAI else " (01 에서는 파인튜닝 QA 생성 때만)")},
    "HuggingFace": {"값": "로그인됨" if _hf else "로그인 안 됨", "출처": "hf auth login" if _hf else "-",
                    "용도": "모델 다운로드 (KURE-v1·Qwen2.5-VL 은 공개 모델이라 로그인 없이도 받아짐)"},
}).T)
print("③ 토큰 검증 (각 최대 15초)", flush=True)
tok_check = verify_tokens("gpt-5-mini")
print("검증:", tok_check)
if NEED_OPENAI and any(e in tok_check.get("OPENAI_API_KEY", "") for e in ("AuthenticationError", "PermissionDeniedError", "NotFoundError")):
    raise RuntimeError(f"OpenAI 키를 쓸 수 없습니다: {tok_check['OPENAI_API_KEY']} · 현재 키 {mask(os.environ.get('OPENAI_API_KEY'))}"
                       " → .env 의 OPENAI_API_KEY 확인 후 이 셀을 다시 실행하세요")

print("④ GPU 점검", flush=True)
REQUIRE_GPU = False         # 생성은 API, GPU(L4)는 질문 임베딩에만 사용 — 없으면 CPU
HW = detect()
from rag.hw import hw_table
display(hw_table(HW).rename(columns={"값": "지금 사용 중인 CPU · GPU · 버전"}))
if HW.get("gpu"):
    print(("✅ " if HW.get("kernel_ok") else "❌ ") + str(HW.get("kernel_msg")))
print(gpu_mem())
if REQUIRE_GPU:
    require_gpu(HW)
print("⑤ 메모리 점검", flush=True)
# ⑤ 메모리 점검 — VM RAM 16GB: 다른 노트북 커널이 남아 있으면 메모리가 모자라 리눅스가 커널을 강제 종료(OOM)합니다
CLOSE_OTHER_KERNELS = False   # True: 이 노트북 말고 떠 있는 다른 Jupyter 커널을 종료해 RAM·GPU 확보 (그 노트북들의 변수는 사라짐)
import psutil, subprocess
def _gb(b): return round(b / 1024**3, 1)
_me = psutil.Process()
_gpu = {}
try:
    _o = subprocess.run(["nvidia-smi", "--query-compute-apps=pid,used_memory", "--format=csv,noheader,nounits"],
                        capture_output=True, text=True, timeout=10).stdout
    for _ln in _o.strip().splitlines():
        _p, _m = [x.strip() for x in _ln.split(",")]
        _gpu[int(_p)] = round(int(_m) / 1024, 1)
except Exception:
    pass
_others = []
for _p in psutil.process_iter(["pid", "cmdline", "memory_info", "create_time"]):
    try:
        _cl = _p.info["cmdline"] or []
        if (_p.info["pid"] != _me.pid and _cl and "python" in os.path.basename(_cl[0]).lower()
                and any("ipykernel" in a for a in _cl[1:])):          # 다른 노트북 커널 (python -m ipykernel_launcher …)
            _others.append(_p)
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        pass
if _others:
    display(pd.DataFrame([{"PID": p.pid, "RAM(GB)": _gb(p.info["memory_info"].rss), "GPU(GB)": _gpu.get(p.pid, 0.0),
                           "켜진 지(분)": int((time.time() - p.info["create_time"]) / 60)} for p in _others])
            .set_index("PID").rename_axis("다른 Jupyter 커널"))
    if CLOSE_OTHER_KERNELS:
        for p in _others:
            p.terminate()
        _gone, _alive = psutil.wait_procs(_others, timeout=8)
        for p in _alive:
            p.kill()
        print(f"🧹 다른 커널 {len(_others)}개 종료 → RAM·GPU 반환", flush=True)
        time.sleep(2)
    else:
        print(f"⚠️ 다른 Jupyter 커널 {len(_others)}개가 RAM {sum(_gb(p.info['memory_info'].rss) for p in _others):.1f}GB · "
              f"GPU {sum(_gpu.get(p.pid, 0) for p in _others):.1f}GB 를 쓰고 있습니다 → 메모리 부족으로 커널이 강제 종료될 수 있음\n"
              f"   끄는 법: 해당 노트북에서 커널 종료(Shut Down) · 터미널 'kill {' '.join(str(p.pid) for p in _others)}' · "
              "또는 위 CLOSE_OTHER_KERNELS = True 후 이 셀 재실행", flush=True)
_vm, _sw = psutil.virtual_memory(), psutil.swap_memory()
print(f"RAM 사용 가능 {_gb(_vm.available)} / {_gb(_vm.total)}GB · 이 커널 {_gb(_me.memory_info().rss)}GB · "
      f"스왑 {_gb(_sw.total)}GB · {gpu_mem()}", flush=True)
if _vm.available < 4 * 1024**3:
    print("⚠️ 사용 가능한 RAM 이 4GB 미만입니다 → 다른 커널을 끄거나, 노트북은 한 번에 하나만 실행하세요", flush=True)
if _sw.total == 0:
    print("ℹ️ 스왑이 없습니다. 메모리가 잠깐 모자랄 때 커널이 바로 죽지 않게 하려면 VM 터미널에서 한 번:\n"
          "   sudo fallocate -l 8G /swapfile && sudo chmod 600 /swapfile && sudo mkswap /swapfile && sudo swapon /swapfile", flush=True)
print(f"⏱ 총 {time.time() - T0:.1f}s")

① torch 로딩…


   완료 6.8s
② 토큰 — OpenAI: .env / HuggingFace: 로그인 정보(hf auth login) 사용, HF_TOKEN 입력창 없음
✔ .env 읽음: /home/spai1322/rfg_rag/.env  (키: OPENAI_API_KEY)


,값,출처,용도
OPENAI_API_KEY,sk-pr…6mQA,.env,gpt-5-mini 생성·RAGAS 심판
HuggingFace,로그인 안 됨,-,모델 다운로드 (KURE-v1·Qwen2.5-VL 은 공개 모델이라 로그인 없이도 ...


③ 토큰 검증 (각 최대 15초)
… OpenAI 키 확인 중
  (4.9s)
검증: {'OPENAI_API_KEY': '✅ 유효 · gpt-5-mini 접근 가능', 'HF_TOKEN': '— 미설정 (공개 모델은 토큰 없이도 받을 수 있음)'}
④ GPU 점검
… GPU 커널 점검 중 (첫 실행은 CUDA 초기화로 수 초~수십 초)


,지금 사용 중인 CPU · GPU · 버전
CPU,Intel(R) Xeon(R) CPU @ 2.20GHz
CPU 코어·스레드,물리 2코어 / 논리 4스레드
RAM,15.6 GB
GPU,NVIDIA L4
GPU 메모리,22.0 GB
GPU 아키텍처,sm_89 · bf16 지원
NVIDIA 드라이버,580.178.04
PyTorch,2.11.0+cu128
CUDA (torch 빌드),12.8
cuDNN,9.19.0


✅ CUDA 커널 실행 OK (fp16/bf16 matmul, softmax) (0.9s)
GPU 메모리: 이 프로세스 0.01GB(예약 0.02GB) · 장치 여유 18.9 / 22.0GB
⑤ 메모리 점검


,RAM(GB),GPU(GB),켜진 지(분)
다른 Jupyter 커널,,,
1525568,2.7,2.9,45


⚠️ 다른 Jupyter 커널 1개가 RAM 2.7GB · GPU 2.9GB 를 쓰고 있습니다 → 메모리 부족으로 커널이 강제 종료될 수 있음
   끄는 법: 해당 노트북에서 커널 종료(Shut Down) · 터미널 'kill 1525568' · 또는 위 CLOSE_OTHER_KERNELS = True 후 이 셀 재실행
RAM 사용 가능 4.3 / 15.6GB · 이 커널 1.2GB · 스왑 0.0GB · GPU 메모리: 이 프로세스 0.01GB(예약 0.02GB) · 장치 여유 18.9 / 22.0GB
ℹ️ 스왑이 없습니다. 메모리가 잠깐 모자랄 때 커널이 바로 죽지 않게 하려면 VM 터미널에서 한 번:
   sudo fallocate -l 8G /swapfile && sudo chmod 600 /swapfile && sudo mkswap /swapfile && sudo swapon /swapfile
⏱ 총 14.2s


In [2]:
# 0-2) ★★ 생성 · 검색 · 평가 파라미터 (학습 없이 바로 반영) ★★
WORK_DIR = "artifacts"                       # 01 노트북의 WORK_DIR 과 같아야 함

# ─────────────────────────────────────────── ⚖️ 하이브리드 검색 (BM25 : Dense) · top-k
TOP_K        = 5
DENSE_WEIGHT = 0.8                           # Dense 비율 (BM25 = 1 − 0.6 = 0.4)
FUSION       = "score"                       # score(min-max 가중합) | rrf
CANDIDATE_K, RRF_K = 50, 60
HNSW_EF_SEARCH = 128                         # faiss index_type="hnsw" 일 때만 (재구축 불필요)
ROUTING = dict(enabled=True, auto_extract_orgs=True, followup_query="none", orgs={})   # 질문 속 기관명 → 해당 문서로 범위 한정 (사전 없음)

# ─────────────────────────────────────────── 🤖 LLM (답변 생성) — 토큰 절약 핵심 설정
#   LLM_MODEL       : "gpt-5-mini"  (추론 모델, 입력 $0.25 · 출력 $2.00 / 1M 토큰)
#                     "gpt-6-luna"  (가장 싸고 빠른 GPT-6, 입력 $0.10 · 출력 $0.50 / 1M — REASONING_EFFORT="none" 이면 추론 없이 답변)
#   REASONING_EFFORT: none | minimal | low | medium | high | xhigh | max
#                     낮을수록 '생각(추론) 토큰'이 줄어 싸고 빠름. none = 추론 토큰 0 (gpt-6-luna 등 지원 모델만)
#                     모델이 지원하지 않는 값이면 가장 가까운 낮은 값으로 자동 조정 (예: gpt-5-mini 에 none → minimal, xhigh/max → high)
#   ※ 추론 토큰은 '출력 토큰'으로 과금됩니다. 추론 강도 none 일 때만 TEMPERATURE·TOP_P 가 적용됩니다.
LLM_MODEL = "gpt-5-mini"          # 또는 "gpt-6-luna"
REASONING_EFFORT = "low"  
TEMPERATURE      = 0.2
TOP_P            = 1.0
MAX_TOKENS       = 2048       # 답변 1개 출력 상한 (추론 토큰 포함) — 낮추면 비용 상한도 내려감
ALLOW_GENERAL    = True       # True: 문서에 없으면 '[문서 외 일반지식]' 표시 후 답변
HISTORY_TURNS    = 3          # 02 는 질문마다 새 대화라 영향 없음 (03 챗봇에서만 사용)

# ─────────────────────────────────────────── 🔎 ① 질문 해석 — GPT 가 질문을 '검색용 질문'으로 다시 씀 (오타·한영 자판·한자·영어·줄임말)
#   원래 질문 + 다시 쓴 질문 N개로 각각 검색 → RRF 로 합침
QUERY_REWRITE            = True
REWRITE_MODEL            = "gpt-5-mini"     # 예) "gpt-6-luna" (더 쌈 · REWRITE_REASONING_EFFORT="none" 가능)
REWRITE_REASONING_EFFORT = "low"            # ★ 질문을 해석할 때 얼마나 '생각'할지: none | minimal | low | medium | high | xhigh | max
                                            #   low = 빠르고 쌈(일반 오타·자판) · high = 꼬인 질문·후속 질문도 꼼꼼히 (추론 토큰↑ = 비용↑)
REWRITE_N                = 2                # 다시 쓴 검색 질문 개수
REWRITE_MAX_TOKENS       = 2048             # 해석 1회 출력 상한 (추론 토큰 포함 — high 면 넉넉히)
REWRITE_HISTORY_TURNS    = 2                # 후속 질문('그럼 그 사업은?') 해석에 참고할 직전 대화 턴 수 (03 챗봇)
ANSWER_WITH_REWRITE      = True             # ★ True: 답변 GPT 에도 '검색용 질문'(1순위)을 원래 질문과 함께 전달 → 오타·자판·줄임말·후속 질문을 바로잡은 뜻으로 답함
                                            #   False: 답변 GPT 에는 원래 질문만 (다시 쓴 질문은 검색에만)
# ─────────────────────────────────────────── 🎯 ③ 리랭커 — 후보를 질문과 1:1 로 다시 채점해 진짜 관련 있는 TOP_K 만 (내 GPU · API 비용 0)
RERANK            = True
RERANK_MODEL      = "BAAI/bge-reranker-v2-m3"   # 다국어(한국어 포함) CrossEncoder · 첫 실행 때 약 2.3GB 다운로드
RERANK_CANDIDATES = 30                          # 리랭커에 넣을 후보 수 (멀티 쿼리 RRF 상위 N개)

# ─────────────────────────────────────────── ⚖️ RAGAS 심판(Judge) — 평가 토큰의 대부분이 여기서 나감
JUDGE_MODEL            = "gpt-5-mini"     # 예) "gpt-6-luna" + JUDGE_REASONING_EFFORT="none" → 평가 비용 크게 절감
JUDGE_REASONING_EFFORT = "none"           # none | minimal | low | medium | high | xhigh | max (자동 조정 동일)
JUDGE_MAX_TOKENS       = 4096

# ─────────────────────────────────────────── 📊 평가 방식
#   EVAL_MODE = "free" : 토큰 0 — 골든 데이터셋(0-5)으로 로컬 계산 (Key Fact 답변/근거 · Semantic Sim)
#               "llm"  : RAGAS (심판 LLM 호출, 질문당 약 15~20회)
#               "both" : 둘 다
EVAL_MODE = "both"
RAGAS_CFG = dict(metrics=["faithfulness", "answer_relevancy", "context_precision", "answer_correctness"],
                 judge_model=JUDGE_MODEL, judge_reasoning_effort=JUDGE_REASONING_EFFORT, judge_max_tokens=JUDGE_MAX_TOKENS)
PRICES = {"gpt-5-mini": (0.25, 2.00), "gpt-5-nano": (0.05, 0.40), "gpt-6-luna": (0.10, 0.50), "gpt-5.6-luna": (0.20, 1.20), "gpt-4.1-mini": (0.40, 1.60), "gpt-4.1": (2.00, 8.00)}   # (입력, 출력) $/1M 토큰 — 비용 표시용, 가격이 바뀌면 수정
def usd(p_in, p_out, model):
    """토큰 → 달러 (추론 토큰은 출력 토큰에 포함되어 과금). 가격표에 없는 모델이면 None"""
    p = next((v for k, v in sorted(PRICES.items(), key=lambda kv: -len(kv[0])) if str(model).lower().startswith(k.lower())), None)
    return None if p is None else (p_in or 0) / 1e6 * p[0] + (p_out or 0) / 1e6 * p[1]
def fmt_usd(x):
    return "—" if x is None else (f"${x:,.4f}" if x >= 0.01 else f"${x:.6f}")

# ============ 적용: 01 의 build_config.json(파싱·청크·임베딩·FAISS 설정 + 인덱스 경로) + 위 값 ============
from rag.config import load_config, set_by_path
from rag.hw import profile_overrides, profile_name
from rag.pipeline import Pipeline
from rag import notebook_checkpoint as nb
pd.set_option("display.max_colwidth", 120)
CFG = load_config()
CFG["retrieval"].update(top_k=TOP_K, dense_weight=DENSE_WEIGHT, fusion=FUSION, candidate_k=CANDIDATE_K, rrf_k=RRF_K)
CFG["routing"].update(ROUTING)
CFG["llm"].update(model=LLM_MODEL, temperature=TEMPERATURE, top_p=TOP_P, max_tokens=MAX_TOKENS,
                  reasoning_effort=REASONING_EFFORT, allow_general=ALLOW_GENERAL, history_turns=HISTORY_TURNS)
CFG["rules"]["enabled"] = False              # 규칙·사전 전처리 없음 — 질문 그대로
CFG["evaluation"].update(RAGAS_CFG)
CFG["retrieval"].update(query_rewrite=dict(enabled=QUERY_REWRITE, model=REWRITE_MODEL, effort=REWRITE_REASONING_EFFORT, n=REWRITE_N,
                                            history_turns=REWRITE_HISTORY_TURNS, answer_with_rewrite=ANSWER_WITH_REWRITE),
                        rerank=dict(enabled=RERANK, model=RERANK_MODEL, candidates=RERANK_CANDIDATES))   # 설정이 바뀌면(답변에 검색용 질문 반영 포함) 02 결과 폴더도 새로

# Windows(로컬)에서 만든 artifacts 를 GCP(리눅스)로 옮겨 온 경우: 저장된 경로의 '\' → '/' 로 바꿔야 파일을 찾음
def fix_windows_paths(work_dir):
    import json as _json
    if os.sep == "\\" or not Path(work_dir).exists():
        return 0
    n = 0
    def conv(x):
        nonlocal n
        if isinstance(x, str) and "\\" in x and ("artifacts" in x or x.startswith(str(work_dir))):
            n += 1
            return x.replace("\\", "/")
        if isinstance(x, dict):
            return {k: conv(v) for k, v in x.items()}
        if isinstance(x, list):
            return [conv(v) for v in x]
        return x
    for name in ("active.json", "build_config.json", "manifest.json"):
        p = Path(work_dir) / name
        if p.exists():
            d = _json.loads(p.read_text(encoding="utf-8"))
            d2 = conv(d)
            if d2 != d:
                p.write_text(_json.dumps(d2, ensure_ascii=False, indent=2), encoding="utf-8")
    return n
_n = fix_windows_paths(WORK_DIR)
if _n:
    print(f"ℹ️ Windows 에서 만든 artifacts 감지 → 경로 {_n}개를 리눅스 형식으로 변환했습니다")
pipe = Pipeline.from_build(CFG, WORK_DIR)    # ← 학습 결과 불러오기 (없으면 01 을 실행하라는 오류)
CFG = pipe.cfg
po = profile_overrides(HW)                   # 질문 임베딩 장치만 현재 PC 에 맞춤
CFG["embedding"].update(device=po["embedding.device"], fp16=po["embedding.fp16"])
CFG["faiss"]["hnsw_ef_search"] = HNSW_EF_SEARCH
bi = pipe.build_info
display(pd.DataFrame({"학습 저장 시각": bi["saved_at"], "임베딩 모델(인덱스)": bi["active"].get("embed_model"),
                      "청크 수": (bi["stats"].get("chunk") or {}).get("chunks"),
                      "chunk_size": bi["cfg"]["chunking"]["chunk_size"], "FAISS": bi["cfg"]["faiss"]["index_type"],
                      "HWP 로더": bi["cfg"]["parsing"].get("hwp_loader"), "generate_only": pipe.generate_only}, index=["01 학습 결과"]).T)
print(f"Dense {DENSE_WEIGHT:.0%} : BM25 {1 - DENSE_WEIGHT:.0%} · top-k {TOP_K} · 답변 {LLM_MODEL}(추론 {REASONING_EFFORT}) · max_tokens {MAX_TOKENS} · 심판 {JUDGE_MODEL}(추론 {JUDGE_REASONING_EFFORT}) · 평가 {EVAL_MODE}")
from rag.llm import effort_table
print(f"🔎 질문 해석 {'켜짐 · ' + REWRITE_MODEL + ' · 추론 ' + REWRITE_REASONING_EFFORT + f' · 검색 질문 {REWRITE_N}개' if QUERY_REWRITE else '꺼짐'}"
      f" · 🎯 리랭커 {RERANK_MODEL + f' (후보 {RERANK_CANDIDATES} → {TOP_K})' if RERANK else '꺼짐'}")
display(effort_table(([("검색 질문 해석", REWRITE_MODEL, REWRITE_REASONING_EFFORT)] if QUERY_REWRITE else [])
                     + [("답변 생성", LLM_MODEL, REASONING_EFFORT), ("RAGAS 심판", JUDGE_MODEL, JUDGE_REASONING_EFFORT)]))

,01 학습 결과
학습 저장 시각,2026-10-12 07:12:23
임베딩 모델(인덱스),nlpai-lab/KURE-v1
청크 수,17695
chunk_size,700
FAISS,flat
HWP 로더,HwpHwpxLoader
generate_only,True


Dense 80% : BM25 20% · top-k 5 · 답변 gpt-5-mini(추론 low) · max_tokens 2048 · 심판 gpt-5-mini(추론 none) · 평가 both
🔎 질문 해석 켜짐 · gpt-5-mini · 추론 low · 검색 질문 2개 · 🎯 리랭커 BAAI/bge-reranker-v2-m3 (후보 30 → 5)


,용도,모델,요청 추론 강도,실제 적용,추론 모델,지원 값,비고
0,검색 질문 해석,gpt-5-mini,low,low,예,"minimal, low, medium, high",
1,답변 생성,gpt-5-mini,low,low,예,"minimal, low, medium, high",
2,RAGAS 심판,gpt-5-mini,none,minimal,예,"minimal, low, medium, high","gpt-5-mini: 추론 강도 'none' 미지원 → 'minimal' 로 호출 (지원: minimal, low, medium, high)"


In [3]:
# 0-3) ★ 질문 — 함정 질문 20개 (8개 제안요청서 기준). 사용자가 챗봇에 직접 입력한 문장 그대로 (전처리·가공·LLM 생성 없음)
#   유형: 오타·띄어쓰기 5 · 한문 섞임 5 · 한영 자판(영문 자판으로 친 한글, 두벌식) 5 · 영어 섞임 5
#   premise=True: "맞죠?" 처럼 틀린 전제를 깔아 둔 질문 → 모델이 전제를 바로잡지 못하면 오답
#   docs : 정답 근거 문서 (파일명에 모두 들어 있어야 하는 조각) → 검색이 정답 문서를 가져왔는지 채점
#   facts: 답변에 꼭 들어가야 할 핵심 사실 ('A|B' = 둘 중 하나) → 0-5 골든 · 2-2 생성 평가 · 01 근거 키워드
TRAP_SET = [
    # ---- 오타 및 띄어쓰기 (5)
    dict(q="인천 일자리플렛폼 ISP 사업예산 얼마에여?", type="오타·띄어쓰기", premise=False,
         doc="인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립용역", docs=[["인천", "일자리플랫폼"]],
         facts=["100,000천원|1억"], trap="플렛폼·얼마에여 오타"),
    dict(q="벤처확인종합관리 시스템고도화 사업기간몇일임?", type="오타·띄어쓰기", premise=False,
         doc="(사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업", docs=[["벤처기업협회", "벤처확인"]],
         facts=["150일"], trap="띄어쓰기 없음"),
    dict(q="파주 종량재봉투 사업 공동수급 되나여?", type="오타·띄어쓰기", premise=False,
         doc="파주도시관광공사_종량제봉투 판매관리 전산시스템 개선사업", docs=[["파주도시관광공사", "종량제"]],
         facts=["허용하지 않|허용되지 않|불가|불허|안 됩", "공동수급"], trap="종량재 오타"),
    dict(q="수자원공사 사고분석솔루션 하자보수보증금율 3%맞죠?", type="오타·띄어쓰기", premise=True,
         doc="한국수자원공사_수도사업장 통합 사고분석솔루션 시범구축 용역", docs=[["한국수자원공사", "사고분석"]],
         facts=["2%"], trap="3%라는 틀린 전제"),
    dict(q="을지대 비교과시스탬 기술가격 비율 9:1이죠?", type="오타·띄어쓰기", premise=True,
         doc="을지대학교_을지대학교 비교과시스템 개발", docs=[["을지대학교", "비교과"]],
         facts=["7:3|70:30|기술70"], trap="9:1은 다른 사업들의 비율(기술 90·가격 10)"),
    # ---- 한문 섞임 (5)
    dict(q="仁川 일자리플랫폼 共同受給 最小持分率은?", type="한문 섞임", premise=False,
         doc="인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립용역", docs=[["인천", "일자리플랫폼"]],
         facts=["5%"], trap="다른 사업은 10%"),
    dict(q="서울시 地圖情報 플랫폼 高度化 事業費는 얼마?", type="한문 섞임", premise=False,
         doc="서울특별시_2024년 지도정보 플랫폼 및 전문활용 연계 시스템 고도화 용역", docs=[["서울특별시", "지도정보"]],
         facts=["493,763천원|493,763,000|4억9,376만"], trap="事業費 한자"),
    dict(q="K-water 事業期間은 着手日로부터 90日이죠?", type="한문 섞임", premise=True,
         doc="한국수자원공사_수도사업장 통합 사고분석솔루션 시범구축 용역", docs=[["한국수자원공사", "사고분석"]],
         facts=["120일"], trap="90일이라는 틀린 전제"),
    dict(q="을지대학교 瑕疵擔保 責任期間은?", type="한문 섞임", premise=False,
         doc="을지대학교_을지대학교 비교과시스템 개발", docs=[["을지대학교", "비교과"]],
         facts=["1년"], trap="瑕疵擔保 한자"),
    dict(q="우즈벡 灌漑시스템 用役期間은 언제까지?", type="한문 섞임", premise=False,
         doc="사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화대응 스마트 관개시스템 구축사업", docs=[["아시아물위원회", "관개"]],
         facts=["2027년 12월 31일|2027.12.31|2027-12-31"], trap="灌漑·用役 한자"),
    # ---- 한영 변환 (5) — 두벌식 자판 기준
    dict(q="dmfwleo tkdjqrlrks 언제까지야?", type="한영 자판", premise=False,                    # 을지대 사업기간
         doc="을지대학교_을지대학교 비교과시스템 개발", docs=[["을지대학교", "비교과"]],
         facts=["2025년 4월 30일|2025.04.30|2025.4.30|2025-04-30"], trap="을지대 사업기간 = dmfwleo tkdjqrlrks"),
    dict(q="vkwn whdfidwpqhdxn 사업예산 알려줘", type="한영 자판", premise=False,                 # 파주 종량제봉투
         doc="파주도시관광공사_종량제봉투 판매관리 전산시스템 개선사업", docs=[["파주도시관광공사", "종량제"]],
         facts=["55,000,000원|5,500만"], trap="파주 종량제봉투 = vkwn whdfidwpqhdxn"),
    dict(q="tjdnftlflqeo 제안 발표시간 몇 분?", type="한영 자판", premise=False,                  # 서울시립대
         doc="서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차 고도화 용역", docs=[["서울시립대학교", "학업성취도"]],
         facts=["30분", "20분", "10분"], trap="서울시립대 = tjdnftlflqeo"),
    dict(q="qpscjghkrdls 1ck dhvms 날짜가 언제야?", type="한영 자판", premise=False,              # 벤처확인 1차 오픈
         doc="(사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업", docs=[["벤처기업협회", "벤처확인"]],
         facts=["7월 10일|07.10|7.10"], trap="벤처확인 1차 오픈 = qpscjghkrdls 1ck dhvms, 2차 오픈과 혼동"),
    dict(q="dlscjs ISP gkehrmq 돼요?", type="한영 자판", premise=False,                           # 인천 ISP 하도급
         doc="인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립용역", docs=[["인천", "일자리플랫폼"]],
         facts=["불허|허용하지 않|허용되지 않|불가|할 수 없|안 됩"], trap="인천 하도급 = dlscjs gkehrmq"),
    # ---- 영어 섞임 (5)
    dict(q="K-water 사고분석솔루션 project budget이 VAT 별도 1억9,503만원 맞아?", type="영어 섞임", premise=True,
         doc="한국수자원공사_수도사업장 통합 사고분석솔루션 시범구축 용역", docs=[["한국수자원공사", "사고분석"]],
         facts=["부가가치세 포함|부가세 포함|VAT 포함|VAT) 포함|세가 포함|VAT가 포함|VAT included|including VAT|includes VAT|inclusive of VAT|VAT-inclusive"], trap="VAT 별도라는 틀린 전제"),
    dict(q="Uzbek irrigation 사업 consortium은 최대 몇 개사, minimum share는 몇 %?", type="영어 섞임", premise=False,
         doc="사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화대응 스마트 관개시스템 구축사업", docs=[["아시아물위원회", "관개"]],
         facts=["5개", "10%"], trap="consortium·share 영어"),
    dict(q="Seoul 지도정보 플랫폼 고도화 사업은 subcontracting 가능해?", type="영어 섞임", premise=False,
         doc="서울특별시_2024년 지도정보 플랫폼 및 전문활용 연계 시스템 고도화 용역", docs=[["서울특별시", "지도정보"]],
         facts=["불허|허용하지 않|허용되지 않|불가|할 수 없|not allowed|not permitted"], trap="subcontracting 영어"),
    dict(q="Through which system should proposals be submitted for the 우즈벡-키르기즈스탄 관개 project?", type="영어 섞임", premise=False,
         doc="사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화대응 스마트 관개시스템 구축사업", docs=[["아시아물위원회", "관개"]],
         facts=["나라장터|e-발주|rfp.g2b.go.kr|g2b"], trap="전체 영어 질문"),
    dict(q="벤처확인 system 고도화 budget 3억5,200만원은 VAT excluded 맞죠?", type="영어 섞임", premise=True,
         doc="(사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업", docs=[["벤처기업협회", "벤처확인"]],
         facts=["부가가치세 포함|부가세 포함|VAT 포함|VAT) 포함|세가 포함|VAT가 포함|VAT included|including VAT|includes VAT|inclusive of VAT|VAT-inclusive"], trap="VAT 별도라는 틀린 전제"),
]
QUESTIONS = [t["q"] for t in TRAP_SET]
KEY_FACTS = {t["q"]: t["facts"] for t in TRAP_SET}

# ★ 평가할 질문 세트 — 둘 다 쓰면 04_e2e_eval 에서 검색(01) · 생성(02) · 챗봇(03) 결과를 질문별로 이어 붙일 수 있음
#   "함정 질문": 위 20개 (근거 문서가 인덱스에 있음)
#   "RFP"     : 01 의 2-5 정답 세트(eval/retrieval_golden.jsonl) — 인덱스 문서에 답이 있는 질문 + 문서 밖 질문(거절 평가)
#   ※ 세트를 바꾸면 결과 폴더가 새로 생겨 답변·RAGAS 를 다시 계산합니다
QUESTION_SETS = ["함정 질문"]       # RFP 정답 세트(R01~R08 · X01~X03)는 제외. 다시 넣으려면 ["함정 질문", "RFP"]
ADD_TRAP_TO_RETRIEVAL_GOLD = True   # True: 함정 질문 20개를 eval/retrieval_golden.jsonl 에도 넣음 → 01 의 2-6 이 검색 단계도 채점 · 03 · 04 가 같은 정답 사용

# 질문마다 독립 실행 — 앞 질문을 기억하지 않음(후속 질문·대화 이어가기 없음)
import importlib
import rag.stage_eval as SE
importlib.reload(SE)
from rag.evaluate import load_eval_set
TRAP_GOLD = [dict(id=f"Q{n:02d}", question=t["q"], docs=t["docs"], keywords=t["facts"], type="in", trap_type=t["type"],
                  false_premise=t["premise"], doc=t["doc"], trap=t["trap"]) for n, t in enumerate(TRAP_SET, 1)]
_saved_gold = SE.load_gold()
RFP_GOLD = [g for g in _saved_gold if not g.get("trap_type")]          # 01 의 RFP 정답 세트 (함정 질문 제외)
if ADD_TRAP_TO_RETRIEVAL_GOLD:
    print("정답 세트 저장:", SE.save_gold(RFP_GOLD + TRAP_GOLD), f"(RFP {len(RFP_GOLD)} + 함정 {len(TRAP_GOLD)})")
_items = []
if "함정 질문" in QUESTION_SETS:
    _items += [dict(id=g["id"], conv=f"chat{n}", set="질문", question=g["question"])   # conv 를 질문마다 다르게 = 매번 새 대화
               for n, g in enumerate(TRAP_GOLD, 1)]
if "RFP" in QUESTION_SETS:
    _items += [dict(id=g["id"], conv=f"chat-{g['id']}", set="RFP", question=g["question"]) for g in RFP_GOLD]
SET_OF = {it["id"]: ("함정 질문" if it["set"] == "질문" else it["set"]) for it in _items}
EVAL_ITEMS = load_eval_set(_items)
print(f"질문 {len(EVAL_ITEMS)}개 (각각 독립 실행) · 세트 " + " · ".join(f"{s} {list(SET_OF.values()).count(s)}개" for s in dict.fromkeys(SET_OF.values())))
display(pd.DataFrame([{"id": g["id"], "유형": g["trap_type"], "틀린 전제": "⚠️" if g["false_premise"] else "", "질문": g["question"],
                       "근거 문서": g["doc"][:40], "핵심 사실": " · ".join(g["keywords"])} for g in TRAP_GOLD])
        .style.hide(axis="index").set_properties(**{"text-align": "left"}))

정답 세트 저장: eval/retrieval_golden.jsonl (RFP 11 + 함정 20)
질문 20개 (각각 독립 실행) · 세트 함정 질문 20개


id,유형,틀린 전제,질문,근거 문서,핵심 사실
Q01,오타·띄어쓰기,,인천 일자리플렛폼 ISP 사업예산 얼마에여?,인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립용역,"100,000천원|1억"
Q02,오타·띄어쓰기,,벤처확인종합관리 시스템고도화 사업기간몇일임?,(사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업,150일
Q03,오타·띄어쓰기,,파주 종량재봉투 사업 공동수급 되나여?,파주도시관광공사_종량제봉투 판매관리 전산시스템 개선사업,허용하지 않|허용되지 않|불가|불허|안 됩 · 공동수급
Q04,오타·띄어쓰기,⚠️,수자원공사 사고분석솔루션 하자보수보증금율 3%맞죠?,한국수자원공사_수도사업장 통합 사고분석솔루션 시범구축 용역,2%
Q05,오타·띄어쓰기,⚠️,을지대 비교과시스탬 기술가격 비율 9:1이죠?,을지대학교_을지대학교 비교과시스템 개발,7:3|70:30|기술70
Q06,한문 섞임,,仁川 일자리플랫폼 共同受給 最小持分率은?,인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립용역,5%
Q07,한문 섞임,,서울시 地圖情報 플랫폼 高度化 事業費는 얼마?,서울특별시_2024년 지도정보 플랫폼 및 전문활용 연계 시스템 고도화 용,"493,763천원|493,763,000|4억9,376만"
Q08,한문 섞임,⚠️,K-water 事業期間은 着手日로부터 90日이죠?,한국수자원공사_수도사업장 통합 사고분석솔루션 시범구축 용역,120일
Q09,한문 섞임,,을지대학교 瑕疵擔保 責任期間은?,을지대학교_을지대학교 비교과시스템 개발,1년
Q10,한문 섞임,,우즈벡 灌漑시스템 用役期間은 언제까지?,사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화대응 스마트 관개시,2027년 12월 31일|2027.12.31|2027-12-31


In [4]:
# 0-4) ★ 정답(reference) — Answer Correctness 계산용. 질문 문장(0-3)은 건드리지 않고 '질문 → 정답'을 따로 적습니다.
#   키는 0-3 의 질문 문장과 똑같아야 합니다. 정답이 없는 질문은 빼거나 "" 로 두면 Answer Correctness 가 '—'(계산 안 함).
#   ※ 정답과 근거는 8개 제안요청서에서 확인한 값입니다. 틀린 전제 질문은 정답이 "아닙니다"로 시작 → 전제를 바로잡아야 정답
REFERENCES = {
    # ---- 오타 및 띄어쓰기
    '인천 일자리플렛폼 ISP 사업예산 얼마에여?':
        '인천일자리플랫폼 정보시스템 구축 ISP 수립용역의 사업예산은 금 100,000천원(1억원) 범위 내입니다.',
    '벤처확인종합관리 시스템고도화 사업기간몇일임?':
        '2024년 벤처확인종합관리시스템 기능 고도화 용역사업의 사업기간은 계약일로부터 150일입니다.',
    '파주 종량재봉투 사업 공동수급 되나여?':
        '안 됩니다. 종량제봉투 판매관리 전산시스템 개선사업은 품질보증과 유지관리를 위해 하도급과 공동수급을 모두 허용하지 않습니다.',
    '수자원공사 사고분석솔루션 하자보수보증금율 3%맞죠?':
        '아닙니다. 수도사업장 통합 사고분석솔루션 시범구축 용역의 하자보수보증금율은 계약금액의 2%이며, 하자담보 책임기간은 준공검사 완료일로부터 1년입니다.',
    '을지대 비교과시스탬 기술가격 비율 9:1이죠?':
        '아닙니다. 을지대학교 비교과시스템 개발 사업의 기술 대 가격 배점 비율은 7:3입니다.',
    # ---- 한문 섞임
    '仁川 일자리플랫폼 共同受給 最小持分率은?':
        '인천일자리플랫폼 정보시스템 구축 ISP 수립용역의 공동수급체 구성원별 최소 지분율은 5% 이상이며, 공동수급체는 5개 이하로 구성합니다.',
    '서울시 地圖情報 플랫폼 高度化 事業費는 얼마?':
        '2024년 지도정보 플랫폼 및 전문활용 연계 시스템 고도화 용역의 사업비는 금 493,763천원(부가가치세 포함)입니다.',
    'K-water 事業期間은 着手日로부터 90日이죠?':
        '아닙니다. 수도사업장 통합 사고분석솔루션 시범구축 용역의 사업기간은 착수일로부터 120일간입니다.',
    '을지대학교 瑕疵擔保 責任期間은?':
        '을지대학교 비교과시스템 개발 사업의 하자담보 책임기간은 사업을 종료한 날(최종산출물을 인도한 날)부터 1년입니다.',
    '우즈벡 灌漑시스템 用役期間은 언제까지?':
        '우즈벡-키르기즈스탄 기후변화대응 스마트 관개시스템 구축사업의 용역기간은 계약체결일로부터 2027년 12월 31일까지입니다.',
    # ---- 한영 변환 (영문 자판으로 친 한글)
    'dmfwleo tkdjqrlrks 언제까지야?':
        '을지대학교 비교과시스템 개발 사업의 사업기간은 계약체결일로부터 2025년 4월 30일까지입니다.',
    'vkwn whdfidwpqhdxn 사업예산 알려줘':
        '파주도시관광공사 종량제봉투 판매관리 전산시스템 개선사업의 사업예산은 금 55,000,000원(부가세 포함)입니다.',
    'tjdnftlflqeo 제안 발표시간 몇 분?':
        '서울시립대학교 학업성취도 다차원 종단분석 통합시스템 1차 고도화 용역의 제안 발표시간은 업체당 30분 내외(제안설명 20분, 질의응답 10분)입니다.',
    'qpscjghkrdls 1ck dhvms 날짜가 언제야?':
        '벤처확인종합관리시스템 기능 고도화 사업의 1차 오픈은 2024년 7월 10일(성과조건부 및 스톡옵션 오픈)이며, 2차 오픈(Grand Open)은 2024년 9월 중입니다.',
    'dlscjs ISP gkehrmq 돼요?':
        '안 됩니다. 인천일자리플랫폼 정보시스템 구축 ISP 수립용역은 하도급을 허용하지 않습니다.',
    # ---- 영어 섞임
    'K-water 사고분석솔루션 project budget이 VAT 별도 1억9,503만원 맞아?':
        '아닙니다. 수도사업장 통합 사고분석솔루션 시범구축 용역의 사업예산 일금 195,030,000원은 부가가치세(VAT) 포함 금액입니다.',
    'Uzbek irrigation 사업 consortium은 최대 몇 개사, minimum share는 몇 %?':
        '우즈벡-키르기즈스탄 스마트 관개시스템 구축사업의 공동수급체는 대표사 포함 5개사 이하이며, 참여 업체의 최소 지분율은 10% 이상입니다(공동이행·분담이행 모두 가능).',
    'Seoul 지도정보 플랫폼 고도화 사업은 subcontracting 가능해?':
        '불가능합니다. 2024년 지도정보 플랫폼 및 전문활용 연계 시스템 고도화 용역은 하도급을 허용하지 않습니다.',
    'Through which system should proposals be submitted for the 우즈벡-키르기즈스탄 관개 project?':
        "제안서는 나라장터 e-발주시스템(http://rfp.g2b.go.kr)을 통해 온라인으로 제출하며, '제안서 제출 매뉴얼'을 참고합니다. "
        "(Proposals are submitted online through the 나라장터 e-발주시스템, http://rfp.g2b.go.kr.)",
    '벤처확인 system 고도화 budget 3억5,200만원은 VAT excluded 맞죠?':
        '아닙니다. 벤처확인종합관리시스템 기능 고도화 용역사업의 소요예산 352,000,000원은 부가가치세(VAT) 포함 금액입니다.',
}
_noref = [it["question"] for it in EVAL_ITEMS if not REFERENCES.get(it["question"])]
_extra = [q for q in REFERENCES if q not in {it["question"] for it in EVAL_ITEMS}]
print(f"정답 있음 {len(EVAL_ITEMS) - len(_noref)}/{len(EVAL_ITEMS)}개" + (f" · 정답 없음(Answer Correctness 제외) {len(_noref)}개: {_noref}" if _noref else ""))
if _extra:
    print(f"⚠️ 0-3 에 없는 질문의 정답 {len(_extra)}개 — 질문 문장이 0-3 과 한 글자라도 다르면 연결되지 않습니다: {_extra}")

정답 있음 20/20개


In [5]:
# 0-5) ⭐ 골든 데이터셋 — 질문(0-3) + 정답(0-4) + 핵심 사실 → eval/golden_set.jsonl  (LLM 호출 없음)
#   한 번 만들어 두면: EVAL_MODE="free" 로 토큰 0 평가 · 03 챗봇이 같은 질문에 정답을 자동으로 붙여 평가
#   핵심 사실: 0-3 의 TRAP_SET 'facts' ('A|B' = 둘 중 하나면 정답) → KEY_FACTS. 고치려면 0-3 의 facts 를 수정
#   쓰려면 아래 줄들의 주석(#)을 지우고 실행하세요
# GOLDEN_PATH = "eval/golden_set.jsonl"
# USE_SAVED_GOLDEN = False     # True: 아래 대신 저장된 파일을 그대로 사용 (파일을 직접 고쳤을 때)
# KEY_FACTS = {t["q"]: t["facts"] for t in TRAP_SET}
# from rag.evaluate import build_golden, save_golden, load_golden
# GOLDEN = load_golden(GOLDEN_PATH) if USE_SAVED_GOLDEN else build_golden(QUESTIONS, REFERENCES, KEY_FACTS)
# if not USE_SAVED_GOLDEN:
#     save_golden(GOLDEN, GOLDEN_PATH)
# GOLD = {g["question"]: g for g in GOLDEN}
# display(pd.DataFrame([{"id": g["id"], "질문": g["question"], "정답": "✔" if g["reference"] else "—",
#                        "핵심 사실": " · ".join(g["key_facts"]), "출처": g["key_facts_source"]} for g in GOLDEN]))
# print(f"골든 데이터셋 {len(GOLDEN)}개 → {GOLDEN_PATH}")

## 1. Generate — 저장된 인덱스로 질문별 답변 (학습·임베딩 없음)
> 👥 **담당:** 1-1b ①④ 질문 해석 · 답변 (`rag/rewrite.py`) · ②③ 검색 · 리랭킹 (`rag/retrieval.py`) · 1-2 · 1-3 평가 · 데이터셋

In [6]:
# 1-1) 인덱스 로드 (FAISS + BM25 + 임베딩 모델 — 질문 임베딩용 추론만) + 생성기
from rag.llm import Generator
retriever = pipe.load_retriever()
generator = Generator(CFG["llm"]["model"], CFG["llm"].get("timeout", 120))
chat = pipe.make_chat(retriever, generator)
print("FAISS 벡터", retriever.store.index.ntotal, "| BM25 청크", len(retriever.by_id), "| 임베딩", pipe.active.get("embed_model"),
      "| 전처리:", "없음(질문 그대로)" if chat.pre is None else "켜짐", "|", gpu_mem())

FAISS 벡터 17695 | BM25 청크 17695 | 임베딩 nlpai-lab/KURE-v1 | 전처리: 없음(질문 그대로) | GPU 메모리: 이 프로세스 0.01GB(예약 0.02GB) · 장치 여유 18.9 / 22.0GB


In [7]:
# 1-1b) 🔎 검색 파이프라인 — ① GPT 질문 해석 → ② (원래 질문 + 다시 쓴 질문) 각각 BM25 + Dense → RRF 병합 → ③ 리랭커
#      → ④ 답변 GPT 에는 [원 질문] + [검색용 질문(GPT 해석 1순위)] 을 함께 보냅니다 (ANSWER_WITH_REWRITE=True)
#      코드는 rag/ 에 있음 — 이 셀은 0-2 설정값으로 조립만 합니다 (02·03 이 같은 코드를 씀)
#        rag/rewrite.py   ① 질문 해석 · ④ 답변에 검색용 질문 전달   (담당: 질문 해석 · 답변 · 챗봇)
#        rag/retrieval.py ② 멀티 쿼리 하이브리드 · ③ 리랭커            (담당: 검색 · 리랭킹)
import importlib
import rag.rewrite, rag.retrieval
importlib.reload(rag.rewrite); importlib.reload(rag.retrieval)     # .py 를 고친 뒤 이 셀만 다시 실행하면 바로 반영
from rag.rewrite import QueryRewriter, install_answer_with_rewrite
from rag.retrieval import SmartRetriever

RW = (QueryRewriter(REWRITE_MODEL, effort=REWRITE_REASONING_EFFORT, n=REWRITE_N, max_tokens=REWRITE_MAX_TOKENS,
                    history_turns=REWRITE_HISTORY_TURNS, price=usd) if QUERY_REWRITE else None)
SR = SmartRetriever(retriever, chat, rewriter=RW, rerank=RERANK, rerank_model=RERANK_MODEL, candidates=RERANK_CANDIDATES)
chat.ret = SR                                   # 이제 chat.ask() 의 검색 = ① → ② → ③
install_answer_with_rewrite(chat, SR, enabled=ANSWER_WITH_REWRITE)   # ④ 답변 GPT 에 검색용 질문 함께 전달

if RERANK:                                      # 디스크·RAM 여유를 확인하고 리랭커를 미리 올림 — 안 되면 리랭커 없이 계속
    _ok, _why = SR.prepare_reranker()
    if _ok:
        print(f"🎯 리랭커 준비: {RERANK_MODEL} ({_why}) · {gpu_mem()}")
    else:
        RERANK = False
        CFG["retrieval"]["rerank"]["enabled"] = False      # 결과 폴더 이름에도 '리랭커 끔'으로 반영
        print(f"⚠️ 리랭커 건너뜀 — {_why} → 리랭커 없이(질문 해석 + 멀티 쿼리 하이브리드) 계속합니다")
print(f"🔎 질문 해석: {'켜짐 · ' + REWRITE_MODEL + ' · 추론 ' + REWRITE_REASONING_EFFORT if QUERY_REWRITE else '꺼짐 (원래 질문만 검색)'}"
      f" · 검색 = 원래 질문 + 다시 쓴 질문 {REWRITE_N if QUERY_REWRITE else 0}개 → RRF → {'리랭커 → ' if RERANK else ''}TOP {TOP_K}")
print(f"🧠 답변 GPT 에 보내는 질문: {'원래 질문 + 검색용 질문(1순위)' if (QUERY_REWRITE and ANSWER_WITH_REWRITE) else '원래 질문만'}"
      + (' (질문 해석이 꺼져 있어 원래 질문만)' if (ANSWER_WITH_REWRITE and not QUERY_REWRITE) else ''))

# ── 15개 질문을 GPT 가 어떻게 해석하는지 먼저 확인 (결과는 저장 → 1-2 에서 같은 질문은 다시 호출하지 않음)
PREVIEW_REWRITE = True
RUN_DIR = pipe.eval_set_dir(EVAL_ITEMS)
os.makedirs(RUN_DIR, exist_ok=True)
SR.attach(os.path.join(RUN_DIR, "rewrites.jsonl"))
if QUERY_REWRITE and PREVIEW_REWRITE:
    _u0 = SR.usage()
    _prev = []
    for it in EVAL_ITEMS:
        rw = SR.rewrite(it["question"])
        _prev.append({"id": it["id"], "질문(입력 그대로)": it["question"],
                      "검색용 질문 (GPT 해석)": " · ".join(rw["queries"]) or f"⚠️ {rw.get('error') or '해석 결과 없음'} → 원래 질문만 검색",
                      "추론 강도": rw.get("effort") or "-", "입력/출력 토큰": f"{rw['prompt_tokens']:,} / {rw['completion_tokens']:,}",
                      "비용": fmt_usd(rw.get("usd")), "": "새로 호출" if rw["new"] else "저장본"})
    display(pd.DataFrame(_prev).set_index("id").style.set_properties(**{"text-align": "left"}))
    _u = SR.usage_since(_u0)
    print(f"💵 이 셀의 GPT 사용 — 질문 해석({REWRITE_MODEL}·{REWRITE_REASONING_EFFORT}) {_u['calls']}회 · 입력 {_u['prompt']:,} · "
          f"출력 {_u['completion']:,}(추론 {_u['reasoning']:,}) 토큰 → {fmt_usd(_u['usd'])}  (저장본 재사용분은 $0)")

⚠️ 리랭커 건너뜀 — RAM 여유 3.1GB — 리랭커를 올리려면 4GB 이상 필요 (다른 노트북 커널을 끄세요) → 리랭커 없이(질문 해석 + 멀티 쿼리 하이브리드) 계속합니다
🔎 질문 해석: 켜짐 · gpt-5-mini · 추론 low · 검색 = 원래 질문 + 다시 쓴 질문 2개 → RRF → TOP 5
🧠 답변 GPT 에 보내는 질문: 원래 질문 + 검색용 질문(1순위)


,질문(입력 그대로),검색용 질문 (GPT 해석),추론 강도,입력/출력 토큰,비용,
id,,,,,,
Q01,인천 일자리플렛폼 ISP 사업예산 얼마에여?,인천 일자리플랫폼 ISP 사업 예산액 · 인천광역시 일자리플랫폼 ISP 사업 예산은 얼마인가,low,374 / 300,$0.000693,새로 호출
Q02,벤처확인종합관리 시스템고도화 사업기간몇일임?,벤처확인 종합관리 시스템 고도화 사업 기간은 몇 일인가? · 벤처확인종합관리 시스템 고도화 사업의 사업 기간(총 일수)은 얼마인가?,low,376 / 121,$0.000336,새로 호출
Q03,파주 종량재봉투 사업 공동수급 되나여?,파주시 종량제 봉투 구매 사업에서 공동수급체(공동수급) 참여 가능 여부 · 파주시 종량제 봉투 공급 사업의 공동수급체 참여 조건 및 입찰 참여 가능성,low,374 / 190,$0.000474,새로 호출
Q04,수자원공사 사고분석솔루션 하자보수보증금율 3%맞죠?,수자원공사 사고분석솔루션 사업의 하자보수 보증금율이 3%로 규정되어 있는지 확인 · 수자원공사 '사고분석솔루션' 사업서류에서 하자보수 보증금율이 3퍼센트로 명시되어 있는지 여부,low,384 / 344,$0.000784,새로 호출
Q05,을지대 비교과시스탬 기술가격 비율 9:1이죠?,을지대학교 비교과 시스템 기술·가격 평가 비율이 9:1 여부 · 을지대학교 비교과 프로그램 시스템 입찰(제안요청서)에서 기술 점수 대 가격 점수 비율이 9:1인지 확인,low,379 / 324,$0.000743,새로 호출
Q06,仁川 일자리플랫폼 共同受給 最小持分率은?,인천 일자리플랫폼 사업 공동수급체의 최소지분율은 무엇인가 · 인천 일자리플랫폼 공동수급 참여 시 최소 지분율,low,376 / 242,$0.000578,새로 호출
Q07,서울시 地圖情報 플랫폼 高度化 事業費는 얼마?,서울시 지도정보 플랫폼 고도화 사업비는 얼마? · 서울시 지도정보 플랫폼 고도화 사업비는 얼마입니까?,low,375 / 172,$0.000438,새로 호출
Q08,K-water 事業期間은 着手日로부터 90日이죠?,K-water 사업기간이 착수일로부터 90일인지 여부 · K-water 사업기간 착수일로부터 90일 확인,low,378 / 301,$0.000696,새로 호출
Q09,을지대학교 瑕疵擔保 責任期間은?,을지대학교 하자담보책임 기간 · 을지대학교 하자 담보 책임 기간,low,375 / 164,$0.000422,새로 호출


💵 이 셀의 GPT 사용 — 질문 해석(gpt-5-mini·low) 20회 · 입력 7,529 · 출력 9,668(추론 8,576) 토큰 → $0.0212  (저장본 재사용분은 $0)


In [8]:
# 1-2) 질문별 답변 생성 (1건마다 저장 → ■ 후 재실행하면 이어서). 질문마다 독립 — 앞 질문을 기억하지 않음
#      검색 = 1-1b 파이프라인(질문 해석 → 멀티 쿼리 하이브리드 → 리랭커)
#      답변 = 답변 GPT 가 [원 질문] + [검색용 질문(GPT 해석 1순위)] 을 함께 받아 답함 (0-2 ANSWER_WITH_REWRITE)
SHOW_ANSWERS = True       # True: 답변 원문도 표시
import os
from IPython.display import Markdown
from rag.evaluate import run_eval_set
from rag.notebook_checkpoint import JsonlStore
RUN_DIR = pipe.eval_set_dir(EVAL_ITEMS)

# ⏱ 워밍업 — 임베딩 모델 · 형태소 분석기(BM25) · FAISS 를 먼저 한 번 돌려 둠 → 첫 질문 응답 시간에 '모델 로딩 시간'이 섞이지 않게 (GPT 호출 없음 · $0)
import json, time, statistics
_t = time.time()
retriever.search("워밍업", top_k=1)
WARMUP_S = round(time.time() - _t, 2)
print(f"⏱ 워밍업 {WARMUP_S}s — 이 시간은 질문별 응답 시간에 넣지 않습니다", flush=True)

# 이전 실행에서 모델 로딩 시간이 섞여 저장된 질문(첫 질문 등)은 답변을 다시 만들어 시간을 다시 잼 (질문당 약 $0.001 + RAGAS 재채점)
RETIME_COLD = True
def _rows_of(path):
    out = []
    if os.path.exists(path):
        for l in open(path, encoding="utf-8"):
            try:
                out.append(json.loads(l))
            except Exception:
                pass
    return out
def _rid(x):
    return x.get("id") or (x.get("value") or {}).get("id") or x.get("_key")
_ans_path = os.path.join(RUN_DIR, "answers.jsonl")
_saved = _rows_of(_ans_path)
_rt = {_rid(x): (x.get("retrieval_s") if "retrieval_s" in x else (x.get("value") or {}).get("retrieval_s")) for x in _saved}
_rt = {k: v for k, v in _rt.items() if isinstance(v, (int, float))}
COLD_IDS = []
if RETIME_COLD and len(_rt) >= 3:
    _med = statistics.median(_rt.values())
    COLD_IDS = [k for k, v in _rt.items() if v > max(5.0, 3 * _med)]
if COLD_IDS:
    print(f"🔁 모델 로딩 시간이 섞인 저장본 {COLD_IDS} (검색 {[round(_rt[k], 1) for k in COLD_IDS]}s · 다른 질문 중앙값 {_med:.1f}s) → 다시 생성해 시간 재측정")
    for _p in [_ans_path] + [os.path.join(RUN_DIR, f) for f in os.listdir(RUN_DIR) if f.startswith("ragas_") and f.endswith(".jsonl")]:
        _keep = [x for x in _rows_of(_p) if _rid(x) not in COLD_IDS]
        with open(_p, "w", encoding="utf-8") as _f:
            _f.writelines(json.dumps(x, ensure_ascii=False) + "\n" for x in _keep)

answers, NEW_IDS = [], set()
_u0 = SR.usage()
h = display(Markdown("⏳ 답변 생성 시작…"), display_id=True)
try:
    with pipe.stop.install_signal_handlers():
        for row, new in run_eval_set(chat, EVAL_ITEMS, JsonlStore(os.path.join(RUN_DIR, "answers.jsonl"), key="id"), pipe.stop):
            answers.append(row)
            if new:
                NEW_IDS.add(row["id"])
            h.update(Markdown(f"답변 생성 **{len(answers)}/{len(EVAL_ITEMS)}** · 방금 {row['id']} ({'새로 생성' if new else '저장본'})"))
            if SHOW_ANSWERS:
                _rw = SR.rw_for(row["question"]) or {}
                _rq = " · ".join(_rw.get("queries") or []) or "(해석 없음 — 원래 질문만 검색)"
                _aq = (row.get("augmentation") or {}).get("답변에 쓴 질문")
                _al = (f"🧠 답변 GPT 가 받은 질문: [원 질문] {row['question']} ➕ [검색용 질문] **{_aq}**" if _aq and _aq != row["question"]
                       else "🧠 답변 GPT 가 받은 질문: 원래 질문만" + ("" if _aq else " (이전 방식 저장본)"))
                display(Markdown(f"**{row['id']}. {row['question']}**  ⏱ {row.get('total_s')}s (검색 {row.get('retrieval_s')}s · 생성 {row.get('llm_s')}s)"
                                 f"\n\n🔎 검색용 질문: {_rq}\n\n{_al}\n\n{row['answer']}\n\n---"))
except KeyboardInterrupt:
    pass
if len(answers) < len(EVAL_ITEMS):
    h.update(Markdown(f"⏸ **중지됨** {len(answers)}/{len(EVAL_ITEMS)} — 셀을 다시 실행하면 이어서 진행합니다."))
else:
    h.update(Markdown(f"✅ 답변 {len(answers)}개 완료 → 아래 2번에서 질문별 RAGAS"))
_nq = sum(1 for r in answers if (r.get("augmentation") or {}).get("검색용 질문 반영"))
print(f"🧠 검색용 질문을 답변에 반영: {_nq}/{len(answers)}건" + ("" if ANSWER_WITH_REWRITE else " (ANSWER_WITH_REWRITE=False — 원래 질문만)"))

# 💵 이 셀의 GPT 사용 (이번 실행에서 새로 호출한 것만 — 저장본 재사용분은 $0)
def _aug(r, k):
    return (r.get("augmentation") or {}).get(k) or 0
_u = SR.usage_since(_u0)
_new = [r for r in answers if r["id"] in NEW_IDS]
ANS_NEW = dict(calls=len(_new), prompt=sum(_aug(r, "prompt_tokens") for r in _new), completion=sum(_aug(r, "completion_tokens") for r in _new),
               reasoning=sum(_aug(r, "reasoning_tokens") for r in _new))
ANS_NEW["usd"] = usd(ANS_NEW["prompt"], ANS_NEW["completion"], LLM_MODEL) or 0
print(f"💵 이 셀의 GPT 사용\n"
      f"   ① 질문 해석({REWRITE_MODEL}·{REWRITE_REASONING_EFFORT}) {_u['calls']}회 · 입력 {_u['prompt']:,} · 출력 {_u['completion']:,}(추론 {_u['reasoning']:,}) → {fmt_usd(_u['usd'])}"
      + ("  (1-1b 에서 이미 해석한 질문은 $0)" if QUERY_REWRITE else "") + "\n"
      f"   ④ 답변 생성({LLM_MODEL}·{REASONING_EFFORT}) {ANS_NEW['calls']}회 · 입력 {ANS_NEW['prompt']:,} · 출력 {ANS_NEW['completion']:,}(추론 {ANS_NEW['reasoning']:,}) → {fmt_usd(ANS_NEW['usd'])}\n"
      f"   합계 {fmt_usd(_u['usd'] + ANS_NEW['usd'])}  (저장본 재사용 {len(answers) - len(_new)}건은 $0)")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

⏱ 워밍업 22.98s — 이 시간은 질문별 응답 시간에 넣지 않습니다


✅ 답변 20개 완료 → 아래 2번에서 질문별 RAGAS

**Q01. 인천 일자리플렛폼 ISP 사업예산 얼마에여?**  ⏱ 3.092s (검색 0.295s · 생성 2.8s)

🔎 검색용 질문: 인천 일자리플랫폼 ISP 사업 예산액 · 인천광역시 일자리플랫폼 ISP 사업 예산은 얼마인가

🧠 답변 GPT 가 받은 질문: [원 질문] 인천 일자리플렛폼 ISP 사업예산 얼마에여? ➕ [검색용 질문] **인천 일자리플랫폼 ISP 사업 예산액**

사업예산은 "금100,000천원(금일억원) 범위내"입니다. [1]

---

**Q02. 벤처확인종합관리 시스템고도화 사업기간몇일임?**  ⏱ 2.028s (검색 0.326s · 생성 1.7s)

🔎 검색용 질문: 벤처확인 종합관리 시스템 고도화 사업 기간은 몇 일인가? · 벤처확인종합관리 시스템 고도화 사업의 사업 기간(총 일수)은 얼마인가?

🧠 답변 GPT 가 받은 질문: [원 질문] 벤처확인종합관리 시스템고도화 사업기간몇일임? ➕ [검색용 질문] **벤처확인 종합관리 시스템 고도화 사업 기간은 몇 일인가?**

사업기간: 계약일로부터 150일 [1]

---

...

**Q17. Uzbek irrigation 사업 consortium은 최대 몇 개사, minimum share는 몇 %?**  ⏱ 3.783s (검색 0.362s · 생성 3.42s)

🔎 검색용 질문: Uzbek irrigation 사업 컨소시엄은 최대 몇 개사로 구성할 수 있으며 참여사별 최소 지분(최소 지분율)은 몇 퍼센트인지 · Uzbek irrigation 사업 컨소시엄의 참여사 수 최대 한도와 참여사별 최소 지분 비율은 얼마인지

🧠 답변 GPT 가 받은 질문: [원 질문] Uzbek irrigation 사업 consortium은 최대 몇 개사, minimum share는 몇 %? ➕ [검색용 질문] **Uzbek irrigation 사업 컨소시엄은 최대 몇 개사로 구성할 수 있으며 참여사별 최소 지분(최소 지분율)은 몇 퍼센트인지**

Uzbek irrigation 사업 컨소시엄은 "공동도급 총 업체수는 대표사를 포함하여 5개사 이하"이며, 참여사별 "최소지분율은 10% 이상"입니다. [1]

---

**Q18. Seoul 지도정보 플랫폼 고도화 사업은 subcontracting 가능해?**  ⏱ 3.921s (검색 0.405s · 생성 3.52s)

🔎 검색용 질문: Seoul 지도정보 플랫폼 고도화 사업에서 하도급(재위탁 포함) 허용 여부와 관련된 규정·조건은 무엇인가? · Seoul 지도정보 플랫폼 고도화 사업의 하도급 허용 기준·절차·제한사항(금지 항목, 사전 승인 필요 여부 등)은 어떻게 되는가?

🧠 답변 GPT 가 받은 질문: [원 질문] Seoul 지도정보 플랫폼 고도화 사업은 subcontracting 가능해? ➕ [검색용 질문] **Seoul 지도정보 플랫폼 고도화 사업에서 하도급(재위탁 포함) 허용 여부와 관련된 규정·조건은 무엇인가?**

정정: 없음.

답변: 본 사업은 하도급을 불허함. [1]

추가참고: 적용·준수할 수 있는 관련 지침들이 열거되어 있으나(예: 하도급거래공정화지침 등), 본 제안요청서에서는 “본 사업은 하도급을 불허 함”으로 명시되어 있어 하도급(재위탁 포함)에 대한 예외 규정이나 허용 조건은 문서에 제시되어 있지 않습니다. [2][1]

---

**Q19. Through which system should proposals be submitted for the 우즈벡-키르기즈스탄 관개 project?**  ⏱ 2.535s (검색 0.329s · 생성 2.21s)

🔎 검색용 질문: 우즈벡-키르기즈스탄 관개 사업의 제안서는 어떤 시스템을 통해 제출해야 합니까? · 우즈벡-키르기즈스탄 관개 사업 제안서 제출은 어떤 제출 시스템을 이용해야 하나요?

🧠 답변 GPT 가 받은 질문: [원 질문] Through which system should proposals be submitted for the 우즈벡-키르기즈스탄 관개 project? ➕ [검색용 질문] **우즈벡-키르기즈스탄 관개 사업의 제안서는 어떤 시스템을 통해 제출해야 합니까?**

우즈벡-키르기즈스탄 관개 사업의 제안서는 나라장터의 e-발주시스템을 통한 온라인 제출입니다. [1]

---

**Q20. 벤처확인 system 고도화 budget 3억5,200만원은 VAT excluded 맞죠?**  ⏱ 5.765s (검색 0.36s · 생성 5.4s)

🔎 검색용 질문: 벤처확인 시스템 고도화 예산 3억 5,200만원은 부가가치세 별도인지 · 벤처확인 시스템 고도화 사업 예산 3억 5,200만원의 부가가치세 포함 여부

🧠 답변 GPT 가 받은 질문: [원 질문] 벤처확인 system 고도화 budget 3억5,200만원은 VAT excluded 맞죠? ➕ [검색용 질문] **벤처확인 시스템 고도화 예산 3억 5,200만원은 부가가치세 별도인지**

컨텍스트 문서에는 예산 3억 5,200만원이 부가가치세 포함인지 여부에 대한 명시가 없습니다. 문서 내 관련 내용이 확인되지 않습니다.

[문서 외 일반지식] 관행적으로 공공 입찰·제안서에서는 금액이 '부가가치세 별도'인지 '부가가치세 포함'인지 명확히 표기합니다. 확인 방법은 다음과 같습니다.
- 제안요청서 본문(예산·금액 표기), 계약서 또는 입찰공고의 금액 표시 문구를 확인하세요. (예: "금액은 부가가치세 별도" 또는 "부가가치세 포함")
- 예산 관련 별지·붙임(예산서, 비용산정표 등)을 확인하세요.
- 발주기관 담당자에게 서면으로 문의하여 확답을 받으세요.

원하시면 제가 문서 전체에서 '부가가치세', 'VAT', '포함', '별도' 같은 키워드를 찾아드리거나, 발주처에 질의할 문장 예시를 작성해 드리겠습니다.

---

🧠 검색용 질문을 답변에 반영: 19/20건
💵 이 셀의 GPT 사용
   ① 질문 해석(gpt-5-mini·low) 0회 · 입력 0 · 출력 0(추론 0) → $0.000000  (1-1b 에서 이미 해석한 질문은 $0)
   ④ 답변 생성(gpt-5-mini·low) 20회 · 입력 34,118 · 출력 7,074(추론 4,352) → $0.0227
   합계 $0.0227  (저장본 재사용 0건은 $0)


In [9]:
# 1-3) ⏱ 질문별 응답 시간 + 🪙 토큰 + 💵 비용 — ① 질문 해석 · ④ 답변 생성 (저장본은 처음 생성했을 때 값)
#      응답 시간에는 1-2 의 워밍업(모델 로딩) 시간이 들어가지 않습니다
TIME_COLS = ["검색(s)", "생성(s)", "전체 응답(s)"]
TOK_COLS = ["해석 토큰", "답변 입력", "답변 출력", "답변 추론"]
USD_COLS = ["해석 $", "답변 $", "합계 $"]
def _aug(r, k):
    return (r.get("augmentation") or {}).get(k)
_rows = []
for r in answers:
    rw = SR.rw_for(r["question"]) or {}
    a_usd = usd(_aug(r, "prompt_tokens") or 0, _aug(r, "completion_tokens") or 0, LLM_MODEL)
    _rows.append({"id": r["id"], "질문": r["question"], "검색용 질문": " · ".join(rw.get("queries") or []),
                  "답변 GPT 질문": (_aug(r, "답변에 쓴 질문") or r["question"]) if _aug(r, "검색용 질문 반영") else "원래 질문만",
                  "검색(s)": r.get("retrieval_s"), "생성(s)": r.get("llm_s"), "전체 응답(s)": r.get("total_s"),
                  "해석 토큰": (rw.get("prompt_tokens") or 0) + (rw.get("completion_tokens") or 0) if rw else None,
                  "답변 입력": _aug(r, "prompt_tokens"), "답변 출력": _aug(r, "completion_tokens"), "답변 추론": _aug(r, "reasoning_tokens"),
                  "해석 $": rw.get("usd") if rw else None, "답변 $": a_usd,
                  "합계 $": (rw.get("usd") or 0) + (a_usd or 0) if (rw or a_usd is not None) else None})
rt = pd.DataFrame(_rows)
_sty = (rt.style.format("{:.2f}", subset=TIME_COLS, na_rep="—").format("{:,.0f}", subset=TOK_COLS, na_rep="—")
        .format(lambda x: fmt_usd(x) if x == x and x is not None else "—", subset=USD_COLS)
        .set_properties(subset=["질문", "검색용 질문", "답변 GPT 질문"], **{"text-align": "left"}))
for _c, _col in (("전체 응답(s)", "#f4a582"), ("합계 $", "#92c5de")):
    if pd.to_numeric(rt[_c], errors="coerce").fillna(0).max() > 0:      # 전부 0 이면 막대 생략 (0 나누기 경고 방지)
        _sty = _sty.bar(subset=[_c], color=_col, vmin=0)
display(_sty)
_num = rt[TIME_COLS + TOK_COLS + USD_COLS].apply(pd.to_numeric, errors="coerce")
rt_stat = _num.agg(["mean", "median", "min", "max", "sum"]).rename(index={"mean": "평균", "median": "중앙값", "min": "최소", "max": "최대", "sum": "합계"})
display(rt_stat.style.format("{:.2f}", subset=TIME_COLS).format("{:,.0f}", subset=TOK_COLS).format(fmt_usd, subset=USD_COLS))
if len(rt):
    slow = rt.loc[pd.to_numeric(rt["전체 응답(s)"], errors="coerce").idxmax()]
    print(f"가장 느린 질문: {slow['id']} {slow['질문']} — {slow['전체 응답(s)']:.2f}s  ·  CPU {HW.get('cpu_model')} · GPU {HW.get('gpu') or '없음'}"
          f"  ·  워밍업(모델 로딩) {globals().get('WARMUP_S', '—')}s 는 제외")
    print(f"💵 질문 {len(rt)}개 전체(저장본 포함): 질문 해석 {fmt_usd(_num['해석 $'].sum())} + 답변 생성 {fmt_usd(_num['답변 $'].sum())} = {fmt_usd(_num['합계 $'].sum())}"
          f"  · 질문당 평균 {fmt_usd(_num['합계 $'].mean())}   (이 셀 자체는 GPT 를 호출하지 않음 · $0)")

,id,질문,검색용 질문,답변 GPT 질문,검색(s),생성(s),전체 응답(s),해석 토큰,답변 입력,답변 출력,답변 추론,해석 $,답변 $,합계 $
0,Q01,인천 일자리플렛폼 ISP 사업예산 얼마에여?,인천 일자리플랫폼 ISP 사업 예산액 · 인천광역시 일자리플랫폼 ISP 사업 예산은 얼마인가,인천 일자리플랫폼 ISP 사업 예산액,0.29,2.80,3.09,674,"1,368",290,256,$0.000693,$0.000922,$0.001615
1,Q02,벤처확인종합관리 시스템고도화 사업기간몇일임?,벤처확인 종합관리 시스템 고도화 사업 기간은 몇 일인가? · 벤처확인종합관리 시스템 고도화 사업의 사업 기간(총 일수)은 얼마인가?,벤처확인 종합관리 시스템 고도화 사업 기간은 몇 일인가?,0.33,1.70,2.03,497,"1,671",150,128,$0.000336,$0.000718,$0.001054
2,Q03,파주 종량재봉투 사업 공동수급 되나여?,파주시 종량제 봉투 구매 사업에서 공동수급체(공동수급) 참여 가능 여부 · 파주시 종량제 봉투 공급 사업의 공동수급체 참여 조건 및 입찰 참여 가능성,파주시 종량제 봉투 구매 사업에서 공동수급체(공동수급) 참여 가능 여부,0.37,1.97,2.34,564,"1,723",186,128,$0.000474,$0.000803,$0.001276
3,Q04,수자원공사 사고분석솔루션 하자보수보증금율 3%맞죠?,수자원공사 사고분석솔루션 사업의 하자보수 보증금율이 3%로 규정되어 있는지 확인 · 수자원공사 '사고분석솔루션' 사업서류에서 하자보수 보증금율이 3퍼센트로 명시되어 있는지 여부,수자원공사 사고분석솔루션 사업의 하자보수 보증금율이 3%로 규정되어 있는지 확인,0.36,3.73,4.09,728,"2,005",306,256,$0.000784,$0.001113,$0.001897
4,Q05,을지대 비교과시스탬 기술가격 비율 9:1이죠?,을지대학교 비교과 시스템 기술·가격 평가 비율이 9:1 여부 · 을지대학교 비교과 프로그램 시스템 입찰(제안요청서)에서 기술 점수 대 가격 점수 비율이 9:1인지 확인,을지대학교 비교과 시스템 기술·가격 평가 비율이 9:1 여부,0.44,2.01,2.45,703,"1,741",213,128,$0.000743,$0.000861,$0.001604
5,Q06,仁川 일자리플랫폼 共同受給 最小持分率은?,인천 일자리플랫폼 사업 공동수급체의 최소지분율은 무엇인가 · 인천 일자리플랫폼 공동수급 참여 시 최소 지분율,인천 일자리플랫폼 사업 공동수급체의 최소지분율은 무엇인가,0.53,2.33,2.87,618,"1,750",162,128,$0.000578,$0.000762,$0.001339
6,Q07,서울시 地圖情報 플랫폼 高度化 事業費는 얼마?,서울시 지도정보 플랫폼 고도화 사업비는 얼마? · 서울시 지도정보 플랫폼 고도화 사업비는 얼마입니까?,서울시 지도정보 플랫폼 고도화 사업비는 얼마?,0.30,2.14,2.44,547,"1,249",202,128,$0.000438,$0.000716,$0.001154
7,Q08,K-water 事業期間은 着手日로부터 90日이죠?,K-water 사업기간이 착수일로부터 90일인지 여부 · K-water 사업기간 착수일로부터 90일 확인,K-water 사업기간이 착수일로부터 90일인지 여부,0.28,2.06,2.34,679,"1,559",163,128,$0.000696,$0.000716,$0.001412
8,Q09,을지대학교 瑕疵擔保 責任期間은?,을지대학교 하자담보책임 기간 · 을지대학교 하자 담보 책임 기간,을지대학교 하자담보책임 기간,0.33,3.99,4.32,539,"1,963",497,256,$0.000422,$0.001485,$0.001906
9,Q10,우즈벡 灌漑시스템 用役期間은 언제까지?,우즈베키스탄 관개 시스템 용역 기간은 언제까지인가? · 우즈베키스탄 관개시스템 사업의 용역 기간 종료일은 언제인가?,우즈베키스탄 관개 시스템 용역 기간은 언제까지인가?,0.26,2.53,2.79,622,"1,853",176,128,$0.000586,$0.000815,$0.001401


,검색(s),생성(s),전체 응답(s),해석 토큰,답변 입력,답변 출력,답변 추론,해석 $,답변 $,합계 $
평균,0.32,3.46,3.78,860,"1,706",354,218,$0.001061,$0.001134,$0.002195
중앙값,0.33,2.88,3.06,676,"1,746",274,192,$0.000695,$0.000992,$0.001697
최소,0.07,1.70,2.03,497,943,150,128,$0.000336,$0.000716,$0.001054
최대,0.53,8.50,8.70,"2,196","2,174","1,039",384,$0.003739,$0.002594,$0.005320
합계,6.31,69.24,75.56,"17,197","34,118","7,074","4,352",$0.0212,$0.0227,$0.0439


가장 느린 질문: Q11 dmfwleo tkdjqrlrks 언제까지야? — 8.70s  ·  CPU Intel(R) Xeon(R) CPU @ 2.20GHz · GPU NVIDIA L4  ·  워밍업(모델 로딩) 22.98s 는 제외
💵 질문 20개 전체(저장본 포함): 질문 해석 $0.0212 + 답변 생성 $0.0227 = $0.0439  · 질문당 평균 $0.002195   (이 셀 자체는 GPT 를 호출하지 않음 · $0)


## 2. 평가 — ① 전체 질문 종합 → ② 질문별 (실시간 표)
> 👥 **담당:** 평가 · 데이터셋 (02 노트북 책임자) — `rag/evaluate.py` `qa_gen.py` `async_utils.py`
- `EVAL_MODE="free"`: **토큰 0** — 골든 데이터셋(0-5)으로 로컬 계산
  - **Key Fact (답변)**: 핵심 사실이 답변에 들어 있는 비율 (≈ 정답 여부)
  - **Key Fact (근거)**: 핵심 사실이 검색된 문단에 들어 있는 비율 (≈ 검색 품질)
  - **Semantic Sim**: 답변과 정답의 의미 유사도 (임베딩 코사인, 내 GPU/CPU)
- `EVAL_MODE="llm"`: RAGAS — Faithfulness · Answer Relevancy · Context Precision · Answer Correctness (심판 LLM 호출)
- `EVAL_MODE="both"`: 둘 다. 표 아래에 **RAGAS 심판이 쓴 토큰과 예상 비용**이 표시됩니다.
- 질문 1건·지표 1개가 끝날 때마다 표가 갱신되고 저장됩니다. ■ 후 다시 실행하면 남은 것만 계산합니다. 같은 설정이면 저장된 점수를 재사용(토큰 0)합니다.
- ② 질문별 표와 `ragas_per_question.csv` 에 응답 시간·토큰이 함께 들어갑니다.

In [10]:
# 2) 평가 — ① 전체 질문 종합 → ② 질문별.   EVAL_MODE: "free"(토큰 0) | "llm"(RAGAS) | "both"
import warnings
warnings.filterwarnings("ignore", message="Unexpected type for token usage")   # langchain-openai 경고(결과와 무관) 숨김
import rag.evaluate as EV
from rag.evaluate import score_rows_iter, free_metrics, FREE_LABELS
from rag.notebook_checkpoint import fingerprint
from rag import llm as L

assert EVAL_MODE in ("free", "llm", "both"), "EVAL_MODE 는 free | llm | both"
# 0-4(정답)·0-5(골든) 셀을 주석 처리하거나 건너뛰어도 실행되도록: 없으면 빈 dict
REFERENCES = globals().get("REFERENCES") or {}
GOLD = globals().get("GOLD") or {}
if not GOLD and EVAL_MODE == "both":
    print("ℹ️ 골든 데이터셋 없음 → 토큰 0 지표(Key Fact·Semantic Sim) 생략 · RAGAS 만 계산")
    EVAL_MODE = "llm"
if not GOLD and EVAL_MODE == "free":
    print("ℹ️ 골든 데이터셋 없음 → 'free' 로 계산할 지표가 없어 평가를 건너뜁니다(토큰 0). RAGAS 를 보려면 0-2 에서 EVAL_MODE='llm'")
if not REFERENCES and not GOLD:
    print("ℹ️ 정답 없음 → Answer Correctness 는 '—'(계산 안 함), Context Precision 은 정답 없는 방식(no-ref)으로 계산")
names = RAGAS_CFG["metrics"] if EVAL_MODE in ("llm", "both") else []
cols_llm = [EV.METRIC_LABELS[m] for m in names]
cols_free = list(FREE_LABELS.values()) if (EVAL_MODE == "both" or (EVAL_MODE == "free" and GOLD)) else []
cols = cols_free + cols_llm
qtext = {r["id"]: r["question"] for r in answers}
rtime = {r["id"]: r.get("total_s") for r in answers}
ref_of = lambda q: (GOLD.get(q) or {}).get("reference") or REFERENCES.get(q) or None   # noqa: E731

# ── 토큰 0 지표 (골든 데이터셋 · 로컬 계산)
free_rows = {}
if cols_free:
    for r in answers:
        g = GOLD.get(r["question"], {})
        free_rows[r["id"]] = free_metrics(r["answer"], r.get("contexts") or [], g.get("reference"), g.get("key_facts"),
                                          retriever.embedder)

def build_df(llm_df=None):
    d = pd.DataFrame([{"id": r["id"], "질문": r["question"], **free_rows.get(r["id"], {})} for r in answers])
    for c_ in cols_free:
        d[c_] = d.get(c_)
    if cols_llm:
        if llm_df is not None and len(llm_df):
            d = d.merge(llm_df[["id"] + cols_llm], on="id", how="left")
        else:
            for c_ in cols_llm:
                d[c_] = None
    d["응답(s)"] = d["id"].map(rtime)
    return d[["id", "질문"] + cols + ["응답(s)"]]

def overall(df):
    recs = []
    for c_ in cols:
        v = pd.to_numeric(df[c_], errors="coerce").dropna()
        recs.append({"지표": c_, "평균": v.mean() if len(v) else None, "중앙값": v.median() if len(v) else None,
                     "최소": v.min() if len(v) else None, "최대": v.max() if len(v) else None,
                     "계산된 질문": f"{len(v)}/{len(answers)}", "종류": "토큰 0" if c_ in cols_free else "RAGAS(LLM)"})
    means = [r_["평균"] for r_ in recs if r_["평균"] is not None and r_["평균"] == r_["평균"]]
    recs.append({"지표": "★ 종합 (지표 평균)", "평균": sum(means) / len(means) if means else None, "중앙값": None,
                 "최소": None, "최대": None, "계산된 질문": f"{len(means)}/{len(cols)} 지표", "종류": ""})
    return pd.DataFrame(recs).astype({"평균": float, "중앙값": float, "최소": float, "최대": float})

num = ["평균", "중앙값", "최소", "최대"]
def show(df, summ):
    h_sum.update(summ.style.format("{:.3f}", subset=num, na_rep="").background_gradient(subset=["평균"], cmap="RdYlGn", vmin=0, vmax=1)
                 .set_properties(subset=["지표"], **{"text-align": "left"}))
    h_tab.update(df.style.format("{:.3f}", subset=cols, na_rep="—").format("{:.2f}", subset=["응답(s)"], na_rep="—")
                 .background_gradient(subset=cols, cmap="RdYlGn", vmin=0, vmax=1).set_properties(subset=["질문"], **{"text-align": "left"}))

display(Markdown(f"### ① 전체 질문 종합 ({EVAL_MODE})"))
h_sum = display(pd.DataFrame(), display_id=True)
display(Markdown("### ② 질문별"))
h_status = display(Markdown("⏳ 준비 중…"), display_id=True)
h_tab = display(pd.DataFrame(), display_id=True)
ragas_df = build_df()
ragas_summary = overall(ragas_df)
show(ragas_df, ragas_summary)
L.reset_judge_usage()
if names:
    rows_for_eval = [{**r, "reference": ref_of(r["question"])} for r in answers]   # 질문은 그대로, 정답만 붙여 평가
    store_r = JsonlStore(os.path.join(RUN_DIR, f"ragas_{fingerprint(names, CFG['evaluation'], {r['question']: ref_of(r['question']) or '' for r in answers})}.jsonl"), key="id")
    try:
        with pipe.stop.install_signal_handlers():
            for df, _, status in score_rows_iter(rows_for_eval, names, CFG["evaluation"], retriever.embedder, store_r, pipe.stop):
                h_status.update(Markdown(f"**{status}** · 심판 호출 {L.JUDGE_USAGE['calls']}회"))
                if len(df):
                    ragas_df = build_df(df)
                    ragas_summary = overall(ragas_df)
                    show(ragas_df, ragas_summary)
    except KeyboardInterrupt:
        h_status.update(Markdown("⏸ **중지됨** — 셀을 다시 실행하면 남은 질문만 평가합니다."))
else:
    h_status.update(Markdown("✅ 토큰 0 평가 완료 (RAGAS 심판 호출 없음)"))

ju = dict(L.JUDGE_USAGE)
JUDGE_NEW = dict(calls=ju["calls"], prompt=ju["prompt_tokens"], completion=ju["completion_tokens"], reasoning=ju["reasoning_tokens"],
                 usd=usd(ju["prompt_tokens"], ju["completion_tokens"], JUDGE_MODEL) or 0)
print(f"💵 이 셀의 GPT 사용 — RAGAS 심판({JUDGE_MODEL}·{JUDGE_REASONING_EFFORT}) {JUDGE_NEW['calls']}회 · 입력 {JUDGE_NEW['prompt']:,} · "
      f"출력 {JUDGE_NEW['completion']:,}(추론 {JUDGE_NEW['reasoning']:,}) 토큰 → {fmt_usd(JUDGE_NEW['usd'])}  (저장된 점수 재사용분은 $0)")

out_csv = Path(WORK_DIR) / "ragas_per_question.csv"
ragas_df.drop(columns=["응답(s)"]).merge(pd.DataFrame([{
    "id": r["id"], "검색(s)": r.get("retrieval_s"), "생성(s)": r.get("llm_s"), "전체 응답(s)": r.get("total_s"),
    "입력 토큰": (r.get("augmentation") or {}).get("prompt_tokens"), "출력 토큰": (r.get("augmentation") or {}).get("completion_tokens"),
    "추론 토큰": (r.get("augmentation") or {}).get("reasoning_tokens"), "답변": r["answer"], "정답": ref_of(r["question"]) or "",
    "핵심 사실": " · ".join((GOLD.get(r["question"]) or {}).get("key_facts") or []),
    "검색용 질문": " · ".join((SR.rw_for(r["question"]) or {}).get("queries") or []),
    "답변 GPT 질문": ((r.get("augmentation") or {}).get("답변에 쓴 질문") or "") if (r.get("augmentation") or {}).get("검색용 질문 반영") else "원래 질문만",
    "해석 $": (SR.rw_for(r["question"]) or {}).get("usd"),
    "답변 $": usd((r.get("augmentation") or {}).get("prompt_tokens") or 0, (r.get("augmentation") or {}).get("completion_tokens") or 0, LLM_MODEL)} for r in answers]),
    on="id", how="left").to_csv(out_csv, index=False, encoding="utf-8-sig")
ragas_summary.to_csv(Path(WORK_DIR) / "ragas_summary.csv", index=False, encoding="utf-8-sig")
print("저장:", out_csv, "·", Path(WORK_DIR) / "ragas_summary.csv")

ℹ️ 골든 데이터셋 없음 → 토큰 0 지표(Key Fact·Semantic Sim) 생략 · RAGAS 만 계산


### ① 전체 질문 종합 (llm)

,지표,평균,중앙값,최소,최대,계산된 질문,종류
0,Faithfulness,0.569,0.535,0.031,1.000,20/20,RAGAS(LLM)
1,Answer Relevancy,0.555,0.610,0.000,0.906,20/20,RAGAS(LLM)
2,Context Precision,0.590,0.736,0.000,1.000,20/20,RAGAS(LLM)
3,Answer Correctness,0.444,0.451,0.100,0.843,20/20,RAGAS(LLM)
4,★ 종합 (지표 평균),0.540,,,,4/4 지표,


### ② 질문별

**✅ 완료 20개** · 심판 호출 260회

,id,질문,Faithfulness,Answer Relevancy,Context Precision,Answer Correctness,응답(s)
0,Q01,인천 일자리플렛폼 ISP 사업예산 얼마에여?,0.667,0.622,1.000,0.780,3.09
1,Q02,벤처확인종합관리 시스템고도화 사업기간몇일임?,0.500,0.584,1.000,0.684,2.03
2,Q03,파주 종량재봉투 사업 공동수급 되나여?,0.750,0.590,0.333,0.843,2.34
3,Q04,수자원공사 사고분석솔루션 하자보수보증금율 3%맞죠?,0.333,0.903,0.583,0.411,4.09
4,Q05,을지대 비교과시스탬 기술가격 비율 9:1이죠?,0.667,0.667,0.450,0.599,2.45
5,Q06,仁川 일자리플랫폼 共同受給 最小持分率은?,0.500,0.585,1.000,0.569,2.87
6,Q07,서울시 地圖情報 플랫폼 高度化 事業費는 얼마?,0.571,0.542,0.000,0.125,2.44
7,Q08,K-water 事業期間은 着手日로부터 90日이죠?,0.667,0.818,1.000,0.168,2.34
8,Q09,을지대학교 瑕疵擔保 責任期間은?,0.625,0.000,0.950,0.482,4.32
9,Q10,우즈벡 灌漑시스템 用役期間은 언제까지?,0.500,0.725,1.000,0.718,2.79


💵 이 셀의 GPT 사용 — RAGAS 심판(gpt-5-mini·none) 260회 · 입력 243,013 · 출력 46,111(추론 0) 토큰 → $0.1530  (저장된 점수 재사용분은 $0)
저장: artifacts/ragas_per_question.csv · artifacts/ragas_summary.csv


## 2-2. 🧠 생성(Generate) 과정 자체 평가 — GPT 없음 ($0)
> 👥 **담당:** 질문 해석 · 답변 — `rag/stage_eval.py`

위 RAGAS 점수에는 **검색이 틀린 탓**과 **생성이 틀린 탓**이 섞여 있습니다. 여기서는 생성 단계만 떼어 봅니다.

| 무엇을 | 어떻게 ($0 · 결정적) |
|---|---|
| ① 단계별로 나눠 보기 | Context Precision · Key Fact(근거) = 검색 / Faithfulness · Answer Relevancy = 생성 / Answer Correctness · Key Fact(답변) = E2E |
| ② **숫자 근거율** | 답변 속 금액·비율·날짜가 검색된 근거에 실제로 있는 비율 → 낮으면 숫자 환각 의심 |
| ② **문장 근거율** | 답변 문장이 근거 문단과 글자를 충분히 공유하는 비율 (`GEN_SENT_THRESHOLD`) |
| ② **핵심 사실** | 근거에 있었나(검색) → 답변에 썼나(생성) → **근거 활용률** = 근거에 있던 핵심 사실 중 답변이 쓴 비율 · **근거 밖 사실률** = 답변의 핵심 사실 중 근거에 없던 비율(일반지식) |
| ② 거절 · 문서 외 표시 · 잘림 | "확인할 수 없습니다" · `[문서 외 일반지식]` · 출력 상한 도달 |
| ③ **조건부 분석** | 검색이 성공한 질문만의 생성 점수 = **생성 단계 자체 실력** (검색 실패가 섞이지 않음) |

In [11]:
# 2-2) 🧠 생성 과정 자체 평가 — 검색이 가져온 근거를 답변이 얼마나 충실히 썼는가.  GPT 호출 없음 ($0)
import importlib
import rag.stage_eval as SE
importlib.reload(SE)
GEN_SENT_THRESHOLD = 0.5      # 답변 문장이 근거 문단과 글자 2-gram 을 이 비율 이상 공유하면 '근거 있음'
TH_RETRIEVAL = 0.5            # 핵심사실(근거) · Context Precision 이 이 값 이상이면 '검색 성공'

# 핵심 사실 — 0-5(골든 데이터셋)를 만들었으면 그것을, 아니면 0-3 의 TRAP_SET facts (A|B = 둘 중 하나). RFP 질문은 01 정답 세트의 근거 키워드
KEY_FACTS = globals().get("KEY_FACTS") or {t["q"]: t["facts"] for t in globals().get("TRAP_SET", [])}
RGOLD = SE.gold_index(list(globals().get("RFP_GOLD") or SE.load_gold()) + list(globals().get("TRAP_GOLD") or []))   # 뒤(함정 질문)가 우선
_chk = SE.check_gold(globals().get("TRAP_GOLD") or [], SE.corpus_sources(retriever))
if len(_chk) and (_chk["맞는 문서 수"] != 1).any():
    print("⚠️ 근거 문서 패턴이 인덱스 파일명과 1:1 로 맞지 않는 질문이 있습니다 → 0-3 의 docs 를 고치세요 (정답 문서 근거 판정에 영향)")
    display(_chk[_chk["맞는 문서 수"] != 1].style.hide(axis="index"))
TMAP = SE.text_source_map(retriever)          # 근거 문단 → 문서 이름 (정답 문서가 근거로 왔는지 확인)

# ① 단계별로 나눠 보기 (위 2) 의 종합표)
def _stage(label):
    m = {"Context Precision": "① 검색", "Context Recall": "① 검색", "Faithfulness": "② 생성", "Answer Relevancy": "② 생성",
         "Answer Correctness": "③ E2E"}
    return m.get(label) or ("① 검색" if "근거" in label else "③ E2E" if ("답변" in label or "Semantic" in label) else "")
_st = ragas_summary[~ragas_summary["지표"].str.startswith("★")].copy()
_st.insert(0, "단계", _st["지표"].map(_stage))
display(Markdown("### ① 평가 지표를 단계별로 (위 2) 의 종합표를 검색 · 생성 · E2E 로 나눔)"),
        _st.sort_values(["단계", "지표"]).style.format("{:.3f}", subset=num, na_rep="")
        .background_gradient(subset=["평균"], cmap="RdYlGn", vmin=0, vmax=1).hide(axis="index"))

# ② 질문별 생성 점검
_llm = ragas_df.set_index("id")[cols] if len(ragas_df) and cols else pd.DataFrame()
gen_rows = []
for r in answers:
    q, aug = r["question"], (r.get("augmentation") or {})
    g = RGOLD.get(r["id"]) or RGOLD.get(SE.qkey(q)) or {}
    ctx = r.get("contexts") or []
    srcs = r.get("sources") or SE.sources_of(ctx, TMAP)
    doc_hit = (any(SE.doc_match(s or "", p) for s in srcs for p in g["docs"])     # 근거 문단의 문서를 모르면 판단 안 함
               if g.get("docs") and any(srcs) else None)
    kf = (GOLD.get(q) or {}).get("key_facts") or KEY_FACTS.get(q) or g.get("keywords")
    c = SE.gen_checks(r["answer"], ctx, key_facts=kf, reference=ref_of(q), completion_tokens=aug.get("completion_tokens"),
                      max_tokens=MAX_TOKENS, sentence_threshold=GEN_SENT_THRESHOLD, doc_hit=doc_hit)
    llm = {k: v for k, v in (_llm.loc[r["id"]].to_dict() if r["id"] in _llm.index else {}).items()}
    row = {"id": r["id"], "세트": SET_OF.get(r["id"], ""), "유형": g.get("trap_type") or SET_OF.get(r["id"], ""),
           "type": g.get("type", "in") if g else "in", "질문": q, **c, **llm}
    if g.get("false_premise"):                     # 틀린 전제 질문: 답변이 전제를 바로잡았나 ("아닙니다 · 사실과 다릅니다 …")
        row["틀린 전제"], row["전제 교정"] = True, SE.premise_corrected(r["answer"])
    ok = SE.retrieval_ok(row, TH_RETRIEVAL)
    row["검색"] = ("🚫 문서 밖 질문" if row["type"] == "ood" else
                  "✅ 검색 성공" if ok else ("❌ 검색 실패" if ok is False else "❔ 판단 불가"))
    gen_rows.append(row)
gen_df = pd.DataFrame(gen_rows)
_gnum = [c_ for c_ in SE.GEN_NUM_COLS + cols_llm if c_ in gen_df]
_show = ["id", "유형", "질문", "검색", "진단"] + (["전제 교정"] if "전제 교정" in gen_df else []) + _gnum + [c_ for c_ in ["정답 문서 근거", "거절", "문서 외 표시", "잘림"] if c_ in gen_df]
def _grad(sty, subset, cmap="RdYlGn"):
    subset = [c_ for c_ in subset if c_ in sty.data and pd.to_numeric(sty.data[c_], errors="coerce").notna().any()]
    return sty.background_gradient(subset=subset, cmap=cmap, vmin=0, vmax=1) if subset else sty
_sty = gen_df[_show].style.format("{:.2f}", subset=_gnum, na_rep="—").set_properties(subset=["질문", "진단"], **{"text-align": "left"})
_sty = _grad(_grad(_sty, [c_ for c_ in _gnum if c_ != "근거 밖 사실률"]), ["근거 밖 사실률"], "RdYlGn_r")
display(Markdown("### ② 질문별 생성 점검 — 진단이 ✅ 가 아닌 질문부터 보세요 (근거 밖 사실률 · 거절은 낮을수록 좋음)"), _sty.hide(axis="index"))
gen_sum = SE.gen_summary(gen_df)
display(Markdown("**생성 점검 요약** (평균 · 비율 · 진단 분포)"),
        gen_sum.style.format("{:.3f}", subset=["값"], na_rep="—").hide(axis="index"))

# ③ 조건부 분석 — 검색이 성공한 질문만의 생성 점수 = 생성 단계 자체 실력
_cmp_cols = [c_ for c_ in ["Faithfulness", "Answer Relevancy", "숫자 근거율", "문장 근거율", "근거 활용률", "근거 밖 사실률",
                           "Answer Correctness", "핵심사실(답변)"] + [c_ for c_ in cols_free if "답변" in c_ or "Semantic" in c_] if c_ in gen_df]
_g = gen_df.copy()
for c_ in _cmp_cols:
    _g[c_] = pd.to_numeric(_g[c_], errors="coerce")
cond = _g.groupby("검색")[_cmp_cols].mean()
cond.insert(0, "질문 수", _g.groupby("검색").size())
display(Markdown("### ③ 검색 성공 / 실패로 나눈 생성 점수 — '✅ 검색 성공' 줄이 생성 단계 자체 실력"),
        _grad(cond.style.format("{:.3f}", subset=_cmp_cols, na_rep="—"), _cmp_cols))
# 함정 유형별 — 어떤 함정(오타 · 한문 · 한영 자판 · 영어 · 틀린 전제)에 약한가
_g["정답 문서 근거"] = _g["정답 문서 근거"].map(lambda v: None if v is None else float(bool(v)))
_tcols = ["정답 문서 근거"] + _cmp_cols
by_type = _g.groupby("유형")[_tcols].mean()
by_type.insert(0, "질문 수", _g.groupby("유형").size())
by_type.insert(1, "진단 ✅ 비율", _g.groupby("유형")["진단"].apply(lambda s: s.astype(str).str.startswith("✅").mean()))
if "전제 교정" in _g:
    by_type.insert(2, "전제 교정률", _g.groupby("유형")["전제 교정"].apply(lambda s: s.dropna().astype(float).mean() if s.notna().any() else None))
_tc = [c_ for c_ in by_type.columns if c_ != "질문 수"]
display(Markdown("### ④ 함정 유형별 — '정답 문서 근거'가 낮으면 검색(질문 해석)이, 그 뒤 지표가 낮으면 생성이 약함"),
        _grad(by_type.style.format("{:.3f}", subset=_tc, na_rep="—"), _tc))
if "전제 교정" in _g:
    _p = _g[_g["틀린 전제"] == True]                 # noqa: E712
    print(f"틀린 전제 질문 {len(_p)}개 중 전제를 바로잡은 답변 {int(_p['전제 교정'].sum())}개 — 못 바로잡은 질문: "
          + (", ".join(_p.loc[~_p["전제 교정"].astype(bool), "id"]) or "없음"))
if gen_df["세트"].nunique() > 1:
    by_set = _g.groupby("세트")[_cmp_cols].mean()
    by_set.insert(0, "질문 수", _g.groupby("세트").size())
    display(Markdown("**세트별** (함정 질문 · RFP 정답 세트)"),
            _grad(by_set.style.format("{:.3f}", subset=_cmp_cols, na_rep="—"), _cmp_cols))

# 저장 → 04_e2e_eval
_rows = []
for r, gr in zip(answers, gen_rows):
    aug, rw = (r.get("augmentation") or {}), (SR.rw_for(r["question"]) or {})
    a_usd = usd(aug.get("prompt_tokens") or 0, aug.get("completion_tokens") or 0, LLM_MODEL)
    g = RGOLD.get(r["id"]) or RGOLD.get(SE.qkey(r["question"])) or {}
    _rows.append({**gr, "gold_id": g.get("id"), "답변": r["answer"], "정답": ref_of(r["question"]) or "",
                  "contexts": r.get("contexts") or [], "sources": r.get("sources") or SE.sources_of(r.get("contexts"), TMAP),
                  "검색용 질문": " · ".join(rw.get("queries") or []),
                  "답변 GPT 질문": (aug.get("답변에 쓴 질문") or "") if aug.get("검색용 질문 반영") else "원래 질문만",
                  "검색(s)": r.get("retrieval_s"), "생성(s)": r.get("llm_s"), "전체 응답(s)": r.get("total_s"),
                  "답변 입력 토큰": aug.get("prompt_tokens"), "답변 출력 토큰": aug.get("completion_tokens"), "답변 추론 토큰": aug.get("reasoning_tokens"),
                  "해석 $": rw.get("usd"), "답변 $": a_usd, "합계 $": (rw.get("usd") or 0) + (a_usd or 0)})
GEN_SUMMARY_DICT = {r_["지표"]: round(r_["값"], 3) for _, r_ in gen_sum.iterrows() if isinstance(r_["값"], float) and r_["종류"] != "진단 분포"}
_saved = SE.save_result("generation_02", {
    "stage": "generation", "notebook": "02_generate_eval", "mode": "single-turn",
    "meta": {"llm_model": LLM_MODEL, "reasoning_effort": REASONING_EFFORT, "temperature": TEMPERATURE, "top_p": TOP_P,
             "max_tokens": MAX_TOKENS, "top_k": TOP_K, "dense_weight": DENSE_WEIGHT, "fusion": FUSION,
             "query_rewrite": QUERY_REWRITE, "rewrite_model": REWRITE_MODEL if QUERY_REWRITE else None,
             "answer_with_rewrite": bool(QUERY_REWRITE and ANSWER_WITH_REWRITE), "rerank": bool(RERANK),
             "retrieval_mode": "hybrid+rerank" if RERANK else "hybrid", "judge_model": JUDGE_MODEL, "eval_mode": EVAL_MODE,
             "question_sets": QUESTION_SETS, "embed_model": pipe.active.get("embed_model"), "run_dir": RUN_DIR,
             "gen_sent_threshold": GEN_SENT_THRESHOLD, "ragas_cols": cols_llm, "free_cols": cols_free},
    "summary": {"ragas": ragas_summary, "generation": gen_sum, "conditional": cond.reset_index()},
    "rows": _rows})
pd.DataFrame(_rows).drop(columns=["contexts"]).to_csv(Path(WORK_DIR) / "gen_eval_02.csv", index=False, encoding="utf-8-sig")
print("저장:", _saved, "·", Path(WORK_DIR) / "gen_eval_02.csv", "· 💵 이 셀의 GPT 사용: 없음 ($0)")

⚠️ 근거 문서 패턴이 인덱스 파일명과 1:1 로 맞지 않는 질문이 있습니다 → 0-3 의 docs 를 고치세요 (정답 문서 근거 판정에 영향)


id,정답 문서 패턴,맞는 문서 수,맞는 문서,상태
Q10,아시아물위원회 + 관개,0,,❌ 없음 — 패턴 수정
Q17,아시아물위원회 + 관개,0,,❌ 없음 — 패턴 수정
Q19,아시아물위원회 + 관개,0,,❌ 없음 — 패턴 수정


### ① 평가 지표를 단계별로 (위 2) 의 종합표를 검색 · 생성 · E2E 로 나눔)

단계,지표,평균,중앙값,최소,최대,계산된 질문,종류
① 검색,Context Precision,0.590,0.736,0.000,1.000,20/20,RAGAS(LLM)
② 생성,Answer Relevancy,0.555,0.610,0.000,0.906,20/20,RAGAS(LLM)
② 생성,Faithfulness,0.569,0.535,0.031,1.000,20/20,RAGAS(LLM)
③ E2E,Answer Correctness,0.444,0.451,0.100,0.843,20/20,RAGAS(LLM)


### ② 질문별 생성 점검 — 진단이 ✅ 가 아닌 질문부터 보세요 (근거 밖 사실률 · 거절은 낮을수록 좋음)

...

### ③ 검색 성공 / 실패로 나눈 생성 점수 — '✅ 검색 성공' 줄이 생성 단계 자체 실력

,질문 수,Faithfulness,Answer Relevancy,숫자 근거율,문장 근거율,근거 활용률,근거 밖 사실률,Answer Correctness,핵심사실(답변)
검색,,,,,,,,,
✅ 검색 성공,15,0.602,0.522,0.900,0.544,1.000,0.091,0.444,0.733
❌ 검색 실패,5,0.473,0.656,1.000,0.807,1.000,0.000,0.445,0.600


### ④ 함정 유형별 — '정답 문서 근거'가 낮으면 검색(질문 해석)이, 그 뒤 지표가 낮으면 생성이 약함

,질문 수,진단 ✅ 비율,전제 교정률,정답 문서 근거,Faithfulness,Answer Relevancy,숫자 근거율,문장 근거율,근거 활용률,근거 밖 사실률,Answer Correctness,핵심사실(답변)
유형,,,,,,,,,,,,
영어 섞임,5,0.800,1.000,0.600,0.578,0.586,0.500,0.700,1.000,0.200,0.519,1.000
오타·띄어쓰기,5,1.000,1.000,1.000,0.583,0.673,1.000,0.700,1.000,0.000,0.663,1.000
한문 섞임,5,0.600,1.000,0.800,0.573,0.534,1.000,0.667,1.000,0.000,0.412,0.600
한영 자판,5,0.200,—,0.600,0.544,0.428,1.000,0.373,1.000,0.000,0.182,0.200


틀린 전제 질문 5개 중 전제를 바로잡은 답변 5개 — 못 바로잡은 질문: 없음
저장: eval/results/generation_02.json · artifacts/gen_eval_02.csv · 💵 이 셀의 GPT 사용: 없음 ($0)


## 3. ✅ 최종 적용 파라미터 (01 학습 설정 + 02 생성·평가 설정)
> 👥 **담당:** PM · 통합

In [12]:
final = nb.final_params(CFG, extra={"01 학습 저장 시각": bi["saved_at"], "임베딩 모델(인덱스)": pipe.active.get("embed_model"),
                                    "프로필(02)": profile_name(HW), "CPU": HW.get("cpu_model"), "GPU": HW.get("gpu") or "없음",
                                    "평균 응답(s)": round(float(pd.Series([r.get("total_s") for r in answers], dtype=float).mean()), 2) if answers else None, "질문 수": len(EVAL_ITEMS), "평가 방식": EVAL_MODE, "답변 추론 강도(요청)": REASONING_EFFORT,
                                    "심판": f"{JUDGE_MODEL} / 추론 {JUDGE_REASONING_EFFORT}",
                                    "질문 해석": f"{REWRITE_MODEL} / 추론 {REWRITE_REASONING_EFFORT} / {REWRITE_N}개" if QUERY_REWRITE else "끔",
                                    "답변에 검색용 질문 반영": bool(QUERY_REWRITE and ANSWER_WITH_REWRITE),
                                    "리랭커": f"{RERANK_MODEL} (후보 {RERANK_CANDIDATES})" if RERANK else "끔", "심판 토큰(이번 실행)": dict(L.JUDGE_USAGE),
                                    "평가 종합(지표 평균)": ragas_summary.set_index("지표")["평균"].round(3).to_dict(),
                                    "질문 세트": globals().get("QUESTION_SETS"),
                                    "생성 자체 평가(2-2)": globals().get("GEN_SUMMARY_DICT")},
                        save_to=str(Path(WORK_DIR) / "final_params.json"))

,구분,파라미터,값
0,로더(라이브러리),pdf_text_loader,PyPDFLoader
1,로더(라이브러리),pypdf,"{""mode"": ""page"", ""extraction_mode"": ""plain"", ""extraction_kwargs"": {}}"
2,로더(라이브러리),pdf_table_loader,PyMuPDFLoader
3,로더(라이브러리),table_settings,{}
4,로더(라이브러리),image_parser,hf-vlm
...,...,...,...
73,추가,리랭커,끔
74,추가,심판 토큰(이번 실행),"{'calls': 260, 'prompt_tokens': 243013, 'completion_tokens': 46111, 'reasoning_tokens': 0}"
75,추가,평가 종합(지표 평균),"{'Faithfulness': 0.569, 'Answer Relevancy': 0.555, 'Context Precision': 0.59, 'Answer Correctness': 0.444, '★ 종합 (지표..."
76,추가,질문 세트,[함정 질문]


## 4. 💵 GPT API 비용 — 단계별 합계
> 👥 **담당:** PM · 통합
이번 실행에서 새로 호출한 비용과, 저장본(이전 실행)까지 포함한 질문 전체 비용을 함께 보여줍니다.

In [13]:
# 4) 💵 GPT API 비용 — ① 질문 해석 · ④ 답변 생성 · RAGAS 심판 (이 셀은 GPT 를 호출하지 않음)
_rw_all = [x for x in (SR.rw_for(r["question"]) for r in answers) if x]
def _aug(r, k):
    return (r.get("augmentation") or {}).get(k) or 0
_rwN = SR.usage()
_ansN = globals().get("ANS_NEW") or dict(calls=0, prompt=0, completion=0, reasoning=0, usd=0)
_jN = globals().get("JUDGE_NEW") or dict(calls=0, prompt=0, completion=0, reasoning=0, usd=0)
_rw_tot = sum(x.get("usd") or 0 for x in _rw_all)
_ans_tot = usd(sum(_aug(r, "prompt_tokens") for r in answers), sum(_aug(r, "completion_tokens") for r in answers), LLM_MODEL) or 0
cost_df = pd.DataFrame([
    {"단계": "① 질문 해석", "모델 · 추론": f"{REWRITE_MODEL} · {REWRITE_REASONING_EFFORT}" if QUERY_REWRITE else "끔",
     "호출(이번)": _rwN["calls"], "입력 토큰(이번)": _rwN["prompt"], "출력 토큰(이번)": _rwN["completion"], "추론 토큰(이번)": _rwN["reasoning"],
     "비용(이번)": _rwN["usd"], "비용(질문 전체·저장본 포함)": _rw_tot},
    {"단계": "④ 답변 생성", "모델 · 추론": f"{LLM_MODEL} · {REASONING_EFFORT}",
     "호출(이번)": _ansN["calls"], "입력 토큰(이번)": _ansN["prompt"], "출력 토큰(이번)": _ansN["completion"], "추론 토큰(이번)": _ansN["reasoning"],
     "비용(이번)": _ansN["usd"], "비용(질문 전체·저장본 포함)": _ans_tot},
    {"단계": "RAGAS 심판", "모델 · 추론": f"{JUDGE_MODEL} · {JUDGE_REASONING_EFFORT}",
     "호출(이번)": _jN["calls"], "입력 토큰(이번)": _jN["prompt"], "출력 토큰(이번)": _jN["completion"], "추론 토큰(이번)": _jN["reasoning"],
     "비용(이번)": _jN["usd"], "비용(질문 전체·저장본 포함)": _jN["usd"]},
])
cost_df.loc[len(cost_df)] = ["합계", "", *cost_df[["호출(이번)", "입력 토큰(이번)", "출력 토큰(이번)", "추론 토큰(이번)", "비용(이번)", "비용(질문 전체·저장본 포함)"]].sum().tolist()]
display(cost_df.style.format("{:,.0f}", subset=["호출(이번)", "입력 토큰(이번)", "출력 토큰(이번)", "추론 토큰(이번)"])
        .format(fmt_usd, subset=["비용(이번)", "비용(질문 전체·저장본 포함)"]).hide(axis="index"))
print("※ ③ 리랭커·임베딩·BM25 는 내 GPU/CPU 에서 계산 → $0.  RAGAS 심판의 '전체'는 이번 실행분만 (저장된 점수의 예전 비용은 기록하지 않음)")

단계,모델 · 추론,호출(이번),입력 토큰(이번),출력 토큰(이번),추론 토큰(이번),비용(이번),비용(질문 전체·저장본 포함)
① 질문 해석,gpt-5-mini · low,20,"7,529","9,668","8,576",$0.0212,$0.0212
④ 답변 생성,gpt-5-mini · low,20,"34,118","7,074","4,352",$0.0227,$0.0227
RAGAS 심판,gpt-5-mini · none,260,"243,013","46,111",0,$0.1530,$0.1530
합계,,300,"284,660","62,853","12,928",$0.1969,$0.1969


※ ③ 리랭커·임베딩·BM25 는 내 GPU/CPU 에서 계산 → $0.  RAGAS 심판의 '전체'는 이번 실행분만 (저장된 점수의 예전 비용은 기록하지 않음)
